In [1]:
# ============================================================
# BuildingClimateML - Full Pipeline (Single File)
# Research Question:
# Are ML models trained on current climate still reliable under
# climate change? Can Transfer Learning recover performance?
# ============================================================

import os
import warnings
import random
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, List, Any

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.utils import resample
from sklearn.decomposition import PCA

import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import shap
import matplotlib.pyplot as plt
import seaborn as sns
from bayes_opt import BayesianOptimization

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

# ============================================================
# 1. CONFIG & REPRODUCIBILITY
# ============================================================
class Config:
    # --- Paths (تغییر دهید) ---
    BASE_DIR = Path("D:/UPM/Machine Learning")          # مسیر اصلی
    CURRENT_FILE = BASE_DIR / "Scenario 1 Present Climate" / "Synthetic_data_sdv_dummies_1826_current climate.csv"
    FILE_2050   = BASE_DIR / "Scenario 2 2050" / "Synthetic_data_future_2050_processed.csv"
    FILE_2080   = BASE_DIR / "Scenario 2 2080" / "Synthetic_data_future_2080_processed.csv"
    OUTPUT_DIR  = BASE_DIR / "Output_New_Pipeline"
    
    # --- Targets ---
    TARGETS = ["t1", "t2", "t3", "t4", "t5"]
    
    # --- Experiment settings ---
    TEST_SIZE = 0.3
    RANDOM_STATE = 42
    N_BOOTSTRAP = 50          # برای سرعت می‌توانید 100 بگذارید
    N_FOLDS = 3               # Nested CV
    BAYES_INIT = 5
    BAYES_ITER = 8
    
    # --- Transfer Learning ---
    TL_FRACTION = 0.3         # فقط ۳۰٪ داده آینده برای fine-tune
    TL_EPOCHS = 40
    TL_LR = 1e-4
    
    # --- DNN ---
    DNN_EPOCHS = 80
    DNN_BATCH = 32
    DNN_PATIENCE = 12

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def set_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(cfg.RANDOM_STATE)

# ============================================================
# 2. DATA LOADING & PREPROCESSING
# ============================================================
def load_or_create_demo_data():
    """اگر فایل واقعی وجود نداشت، داده مصنوعی می‌سازد."""
    if cfg.CURRENT_FILE.exists() and cfg.FILE_2050.exists() and cfg.FILE_2080.exists():
        print("Loading real datasets...")
        df_cur = pd.read_csv(cfg.CURRENT_FILE)
        df_50  = pd.read_csv(cfg.FILE_2050)
        df_80  = pd.read_csv(cfg.FILE_2080)
        return df_cur, df_50, df_80
    
    print("Real files not found → Creating synthetic demo data for structure test...")
    n = 800
    n_features = 25
    np.random.seed(42)
    
    def make_df(shift=0.0, noise=1.0):
        X = np.random.randn(n, n_features) + shift
        # شبیه‌سازی وابستگی غیرخطی + تغییر اقلیم
        y = np.column_stack([
            50 + 3*X[:,0] + 1.5*X[:,1]**2 + noise*np.random.randn(n),          # t1 energy
            20 + 2*X[:,2] - 0.8*X[:,3] + noise*np.random.randn(n),              # t2
            400 + 15*X[:,4] + 5*X[:,5] + noise*np.random.randn(n)*10,           # t3 IAQ
            22 + 0.5*X[:,6] + noise*np.random.randn(n)*0.5,                     # t4
            300 + 8*X[:,7] - 2*X[:,8] + noise*np.random.randn(n)*5              # t5
        ])
        cols = [f"f{i}" for i in range(n_features)] + cfg.TARGETS
        return pd.DataFrame(np.hstack([X, y]), columns=cols)
    
    return make_df(0.0, 1.0), make_df(0.6, 1.3), make_df(1.1, 1.6)

def align_columns(df_ref, df_other):
    ref_feats = [c for c in df_ref.columns if c not in cfg.TARGETS]
    for c in ref_feats:
        if c not in df_other.columns:
            df_other[c] = 0
    return df_other[ref_feats + cfg.TARGETS]

def prepare_xy(df, feature_names=None):
    if feature_names is None:
        feature_names = [c for c in df.columns if c not in cfg.TARGETS]
    X = df[feature_names].values.astype(np.float32)
    y = df[cfg.TARGETS].values.astype(np.float32)
    return X, y, feature_names

# ============================================================
# 3. METRICS + BOOTSTRAP
# ============================================================
def calc_metrics(y_true, y_pred):
    metrics = {}
    for i, t in enumerate(cfg.TARGETS):
        yt, yp = y_true[:, i], y_pred[:, i]
        metrics[t] = {
            "RMSE": np.sqrt(mean_squared_error(yt, yp)),
            "MAE": mean_absolute_error(yt, yp),
            "R2": r2_score(yt, yp),
            "MAPE": np.mean(np.abs((yt - yp) / (np.abs(yt) + 1e-8))) * 100
        }
    # میانگین کلی
    metrics["AVG"] = {
        k: np.mean([metrics[t][k] for t in cfg.TARGETS])
        for k in ["RMSE", "MAE", "R2", "MAPE"]
    }
    return metrics

def bootstrap_ci(model, X, y, n_iter=50, is_dnn=False):
    results = {t: {"RMSE": [], "MAE": [], "R2": [], "MAPE": []} for t in cfg.TARGETS}
    n = len(X)
    for i in range(n_iter):
        idx = resample(np.arange(n), random_state=cfg.RANDOM_STATE + i)
        Xb, yb = X[idx], y[idx]
        if is_dnn:
            pred = model.predict(Xb, verbose=0)
        else:
            pred = model.predict(Xb)
        m = calc_metrics(yb, pred)
        for t in cfg.TARGETS:
            for k in results[t]:
                results[t][k].append(m[t][k])
    
    summary = {}
    for t in cfg.TARGETS:
        summary[t] = {}
        for k in ["RMSE", "MAE", "R2", "MAPE"]:
            arr = np.array(results[t][k])
            summary[t][k] = {
                "mean": np.mean(arr),
                "ci_low": np.percentile(arr, 2.5),
                "ci_high": np.percentile(arr, 97.5)
            }
    return summary

# ============================================================
# 4. MODELS
# ============================================================
def create_dnn(input_dim, lr=0.001):
    model = Sequential([
        Dense(128, activation="relu", input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(len(cfg.TARGETS))
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse")
    return model

def get_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=cfg.DNN_PATIENCE, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6)
    ]

# ============================================================
# 5. EXPERIMENT A: Train only on Current → Test on Future
# ============================================================
def run_experiment_A(X_cur, y_cur, X_50, y_50, X_80, y_80, feature_names):
    print("\n" + "="*60)
    print("EXPERIMENT A: Train on Current only → Test on 2050 & 2080")
    print("="*60)
    
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_cur, y_cur, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE
    )
    
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    X_50_s = scaler.transform(X_50)
    X_80_s = scaler.transform(X_80)
    
    results = {}
    
    # --- Random Forest ---
    print("Training RF...")
    rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=cfg.RANDOM_STATE, n_jobs=-1)
    rf.fit(X_tr_s, y_tr)
    results["RF"] = {
        "current": calc_metrics(y_te, rf.predict(X_te_s)),
        "2050": calc_metrics(y_50, rf.predict(X_50_s)),
        "2080": calc_metrics(y_80, rf.predict(X_80_s)),
        "model": rf,
        "scaler": scaler
    }
    
    # --- XGBoost ---
    print("Training XGBoost...")
    xgb_model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        random_state=cfg.RANDOM_STATE, n_jobs=-1, multi_strategy="multi_output_tree"
    )
    xgb_model.fit(X_tr_s, y_tr)
    results["XGB"] = {
        "current": calc_metrics(y_te, xgb_model.predict(X_te_s)),
        "2050": calc_metrics(y_50, xgb_model.predict(X_50_s)),
        "2080": calc_metrics(y_80, xgb_model.predict(X_80_s)),
        "model": xgb_model,
        "scaler": scaler
    }
    
    # --- DNN ---
    print("Training DNN...")
    dnn = create_dnn(X_tr_s.shape[1])
    dnn.fit(X_tr_s, y_tr, validation_split=0.2, epochs=cfg.DNN_EPOCHS,
            batch_size=cfg.DNN_BATCH, callbacks=get_callbacks(), verbose=0)
    results["DNN"] = {
        "current": calc_metrics(y_te, dnn.predict(X_te_s, verbose=0)),
        "2050": calc_metrics(y_50, dnn.predict(X_50_s, verbose=0)),
        "2080": calc_metrics(y_80, dnn.predict(X_80_s, verbose=0)),
        "model": dnn,
        "scaler": scaler
    }
    
    return results

# ============================================================
# 6. EXPERIMENT B: Transfer Learning
# ============================================================
def run_experiment_B(results_A, X_50, y_50, X_80, y_80):
    print("\n" + "="*60)
    print("EXPERIMENT B: Transfer Learning (fine-tune with limited future data)")
    print("="*60)
    
    results_B = {}
    
    for year, X_fut, y_fut in [("2050", X_50, y_50), ("2080", X_80, y_80)]:
        print(f"\n--- Transfer Learning for {year} ---")
        n_tl = int(len(X_fut) * cfg.TL_FRACTION)
        idx = np.random.choice(len(X_fut), n_tl, replace=False)
        X_tl, y_tl = X_fut[idx], y_fut[idx]
        X_test = np.delete(X_fut, idx, axis=0)
        y_test = np.delete(y_fut, idx, axis=0)
        
        year_res = {}
        
        # RF & XGB: partial refit (simple adaptation)
        for name in ["RF", "XGB"]:
            model = results_A[name]["model"]
            scaler = results_A[name]["scaler"]
            X_tl_s = scaler.transform(X_tl)
            X_test_s = scaler.transform(X_test)
            
            # برای درخت‌ها: ادامه آموزش با داده جدید (warm start تقریبی)
            if name == "RF":
                # RF از scikit-learn warm_start محدود دارد → مدل جدید با داده ترکیبی کوچک
                X_comb = np.vstack([scaler.transform(results_A[name]["scaler"].inverse_transform(
                    results_A[name]["model"].feature_importances_[:len(X_tl)] * 0 + X_tl_s[:min(200, len(X_tl))])), X_tl_s])
                # ساده‌سازی: فقط روی داده TL آموزش مجدد
                new_model = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=cfg.RANDOM_STATE, n_jobs=-1)
                new_model.fit(X_tl_s, y_tl)
            else:
                new_model = xgb.XGBRegressor(
                    n_estimators=150, max_depth=5, learning_rate=0.03,
                    random_state=cfg.RANDOM_STATE, n_jobs=-1, multi_strategy="multi_output_tree"
                )
                new_model.fit(X_tl_s, y_tl)
            
            year_res[name] = calc_metrics(y_test, new_model.predict(X_test_s))
        
        # DNN: واقعی Transfer Learning
        dnn_base = results_A["DNN"]["model"]
        scaler = results_A["DNN"]["scaler"]
        X_tl_s = scaler.transform(X_tl)
        X_test_s = scaler.transform(X_test)
        
        dnn_tl = clone_model(dnn_base)
        dnn_tl.set_weights(dnn_base.get_weights())
        
        # freeze لایه‌های اولیه
        for layer in dnn_tl.layers[:2]:
            layer.trainable = False
        
        dnn_tl.compile(optimizer=Adam(learning_rate=cfg.TL_LR), loss="mse")
        dnn_tl.fit(X_tl_s, y_tl, validation_split=0.2, epochs=cfg.TL_EPOCHS,
                   batch_size=cfg.DNN_BATCH, callbacks=get_callbacks(), verbose=0)
        
        year_res["DNN"] = calc_metrics(y_test, dnn_tl.predict(X_test_s, verbose=0))
        year_res["DNN_model"] = dnn_tl
        
        results_B[year] = year_res
    
    return results_B

# ============================================================
# 7. EXPERIMENT C: Full training on Future
# ============================================================
def run_experiment_C(X_50, y_50, X_80, y_80):
    print("\n" + "="*60)
    print("EXPERIMENT C: Full training from scratch on 2050 & 2080")
    print("="*60)
    
    results_C = {}
    
    for year, X, y in [("2050", X_50, y_50), ("2080", X_80, y_80)]:
        print(f"\n--- Full training on {year} ---")
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE)
        
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)
        
        year_res = {}
        
        # RF
        rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=cfg.RANDOM_STATE, n_jobs=-1)
        rf.fit(X_tr_s, y_tr)
        year_res["RF"] = calc_metrics(y_te, rf.predict(X_te_s))
        
        # XGB
        xgb_m = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05,
                                  random_state=cfg.RANDOM_STATE, n_jobs=-1, multi_strategy="multi_output_tree")
        xgb_m.fit(X_tr_s, y_tr)
        year_res["XGB"] = calc_metrics(y_te, xgb_m.predict(X_te_s))
        
        # DNN
        dnn = create_dnn(X_tr_s.shape[1])
        dnn.fit(X_tr_s, y_tr, validation_split=0.2, epochs=cfg.DNN_EPOCHS,
                batch_size=cfg.DNN_BATCH, callbacks=get_callbacks(), verbose=0)
        year_res["DNN"] = calc_metrics(y_te, dnn.predict(X_te_s, verbose=0))
        
        results_C[year] = year_res
    
    return results_C

# ============================================================
# 8. DOMAIN SHIFT + SHAP
# ============================================================
def analyze_domain_shift(X_cur, X_50, X_80, feature_names):
    print("\nAnalyzing Domain Shift (PCA)...")
    pca = PCA(n_components=2, random_state=cfg.RANDOM_STATE)
    all_X = np.vstack([X_cur[:500], X_50[:500], X_80[:500]])
    emb = pca.fit_transform(StandardScaler().fit_transform(all_X))
    
    labels = (["Current"]*500 + ["2050"]*500 + ["2080"]*500)
    
    plt.figure(figsize=(9, 7))
    sns.scatterplot(x=emb[:,0], y=emb[:,1], hue=labels, palette="Set1", alpha=0.6, s=30)
    plt.title("Domain Shift (PCA) – Current vs 2050 vs 2080")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend()
    plt.tight_layout()
    plt.savefig(cfg.OUTPUT_DIR / "domain_shift_pca.png", dpi=200)
    plt.close()
    print(f"Saved: {cfg.OUTPUT_DIR / 'domain_shift_pca.png'}")

def run_comparative_shap(results_A, X_cur, X_50, feature_names):
    print("\nRunning Comparative SHAP (XGBoost)...")
    try:
        model = results_A["XGB"]["model"]
        scaler = results_A["XGB"]["scaler"]
        
        X_s = scaler.transform(X_cur[:150])
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_s)
        
        # میانگین قدرمطلق برای multi-output
        if isinstance(sv, list):
            mean_abs = np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
        else:
            mean_abs = np.abs(sv).mean(axis=(0, 2)) if sv.ndim == 3 else np.abs(sv).mean(axis=0)
        
        importance = pd.DataFrame({
            "Feature": feature_names,
            "SHAP_Current": mean_abs
        }).sort_values("SHAP_Current", ascending=False)
        
        importance.to_csv(cfg.OUTPUT_DIR / "shap_importance_current.csv", index=False)
        print("SHAP importance saved.")
    except Exception as e:
        print(f"SHAP failed (common with multi-output): {e}")

# ============================================================
# 9. SUMMARY TABLES
# ============================================================
def create_summary_tables(res_A, res_B, res_C):
    rows = []
    
    # Experiment A
    for model_name in ["RF", "XGB", "DNN"]:
        for period in ["current", "2050", "2080"]:
            m = res_A[model_name][period]["AVG"]
            rows.append({
                "Experiment": "A (Current only)",
                "Model": model_name,
                "Period": period,
                "RMSE": round(m["RMSE"], 3),
                "MAE": round(m["MAE"], 3),
                "R2": round(m["R2"], 3),
                "MAPE": round(m["MAPE"], 2)
            })
    
    # Experiment B
    for year in ["2050", "2080"]:
        for model_name in ["RF", "XGB", "DNN"]:
            m = res_B[year][model_name]["AVG"]
            rows.append({
                "Experiment": "B (Transfer Learning)",
                "Model": model_name,
                "Period": year,
                "RMSE": round(m["RMSE"], 3),
                "MAE": round(m["MAE"], 3),
                "R2": round(m["R2"], 3),
                "MAPE": round(m["MAPE"], 2)
            })
    
    # Experiment C
    for year in ["2050", "2080"]:
        for model_name in ["RF", "XGB", "DNN"]:
            m = res_C[year][model_name]["AVG"]
            rows.append({
                "Experiment": "C (Full Future)",
                "Model": model_name,
                "Period": year,
                "RMSE": round(m["RMSE"], 3),
                "MAE": round(m["MAE"], 3),
                "R2": round(m["R2"], 3),
                "MAPE": round(m["MAPE"], 2)
            })
    
    df = pd.DataFrame(rows)
    df.to_csv(cfg.OUTPUT_DIR / "full_comparison.csv", index=False)
    print("\n=== Full Comparison Table ===")
    print(df.to_string(index=False))
    return df

# ============================================================
# 10. MAIN
# ============================================================
def main():
    print("BuildingClimateML Pipeline Started")
    print(f"Output directory: {cfg.OUTPUT_DIR}")
    
    # Load data
    df_cur, df_50, df_80 = load_or_create_demo_data()
    
    # Align columns
    df_50 = align_columns(df_cur, df_50)
    df_80 = align_columns(df_cur, df_80)
    
    # Prepare arrays
    X_cur, y_cur, feature_names = prepare_xy(df_cur)
    X_50, y_50, _ = prepare_xy(df_50, feature_names)
    X_80, y_80, _ = prepare_xy(df_80, feature_names)
    
    print(f"Features: {len(feature_names)} | Samples Current: {len(X_cur)} | 2050: {len(X_50)} | 2080: {len(X_80)}")
    
    # Run experiments
    res_A = run_experiment_A(X_cur, y_cur, X_50, y_50, X_80, y_80, feature_names)
    res_B = run_experiment_B(res_A, X_50, y_50, X_80, y_80)
    res_C = run_experiment_C(X_50, y_50, X_80, y_80)
    
    # Analyses
    analyze_domain_shift(X_cur, X_50, X_80, feature_names)
    run_comparative_shap(res_A, X_cur, X_50, feature_names)
    
    # Summary
    summary_df = create_summary_tables(res_A, res_B, res_C)
    
    # Quick degradation view
    print("\n=== Performance Degradation (Experiment A) ===")
    for model in ["RF", "XGB", "DNN"]:
        r2_cur = res_A[model]["current"]["AVG"]["R2"]
        r2_50  = res_A[model]["2050"]["AVG"]["R2"]
        r2_80  = res_A[model]["2080"]["AVG"]["R2"]
        print(f"{model}: Current R²={r2_cur:.3f} → 2050 R²={r2_50:.3f} (Δ={r2_50-r2_cur:+.3f}) → 2080 R²={r2_80:.3f} (Δ={r2_80-r2_cur:+.3f})")
    
    print("\n" + "="*60)
    print("Pipeline finished successfully!")
    print(f"All results saved to: {cfg.OUTPUT_DIR}")
    print("="*60)

if __name__ == "__main__":
    main()

BuildingClimateML Pipeline Started
Output directory: D:\UPM\Machine Learning\Output_New_Pipeline
Loading real datasets...
Features: 107 | Samples Current: 1826 | 2050: 909 | 2080: 991

EXPERIMENT A: Train on Current only → Test on 2050 & 2080
Training RF...
Training XGBoost...
Training DNN...

EXPERIMENT B: Transfer Learning (fine-tune with limited future data)

--- Transfer Learning for 2050 ---

--- Transfer Learning for 2080 ---

EXPERIMENT C: Full training from scratch on 2050 & 2080

--- Full training on 2050 ---

--- Full training on 2080 ---

Analyzing Domain Shift (PCA)...
Saved: D:\UPM\Machine Learning\Output_New_Pipeline\domain_shift_pca.png

Running Comparative SHAP (XGBoost)...
SHAP failed (common with multi-output): vector-leaf is not yet supported.

=== Full Comparison Table ===
           Experiment Model  Period        RMSE         MAE       R2   MAPE
     A (Current only)    RF current 2322.277000 1456.375000    0.720  36.83
     A (Current only)    RF    2050 3176.705

In [2]:
# ============================================================
# BuildingClimateML - Full Corrected Pipeline
# Research Question:
# Are ML models trained on current climate still reliable under
# climate change? Can Transfer Learning recover performance?
# ============================================================

import os
import warnings
import random
import numpy as np
import pandas as pd
from pathlib import Path
from copy import deepcopy

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition import PCA

import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import shap
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

# ============================================================
# 1. CONFIG
# ============================================================
class Config:
    BASE_DIR = Path(r"D:\UPM\Machine Learning")
    CURRENT_FILE = BASE_DIR / "Scenario 1 Present Climate" / "Synthetic_data_sdv_dummies_1826_current climate.csv"
    FILE_2050   = BASE_DIR / "Scenario 2 2050" / "Synthetic_data_future_2050_processed.csv"
    FILE_2080   = BASE_DIR / "Scenario 2 2080" / "Synthetic_data_future_2080_processed.csv"
    OUTPUT_DIR  = BASE_DIR / "Output_New_Pipeline"

    TARGETS = ["t1", "t2", "t3", "t4", "t5"]
    TEST_SIZE = 0.30
    RANDOM_STATE = 42
    TL_FRACTION = 0.30          # درصد داده آینده برای Transfer Learning
    TL_EPOCHS = 50
    TL_LR = 1e-4
    DNN_EPOCHS = 120
    DNN_BATCH = 32
    DNN_PATIENCE = 15

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(cfg.RANDOM_STATE)

# ============================================================
# 2. DATA LOADING
# ============================================================
def load_data():
    if cfg.CURRENT_FILE.exists() and cfg.FILE_2050.exists() and cfg.FILE_2080.exists():
        print("Loading real datasets...")
        df_cur = pd.read_csv(cfg.CURRENT_FILE)
        df_50  = pd.read_csv(cfg.FILE_2050)
        df_80  = pd.read_csv(cfg.FILE_2080)
        return df_cur, df_50, df_80
    else:
        raise FileNotFoundError("فایل‌های داده پیدا نشدند. مسیرها را در Config بررسی کنید.")

def align_columns(df_ref, df_other):
    ref_feats = [c for c in df_ref.columns if c not in cfg.TARGETS]
    for c in ref_feats:
        if c not in df_other.columns:
            df_other[c] = 0.0
    return df_other[ref_feats + cfg.TARGETS]

def prepare_xy(df, feature_names=None):
    if feature_names is None:
        feature_names = [c for c in df.columns if c not in cfg.TARGETS]
    X = df[feature_names].values.astype(np.float64)
    y = df[cfg.TARGETS].values.astype(np.float64)
    return X, y, feature_names

# ============================================================
# 3. METRICS
# ============================================================
def calc_metrics(y_true, y_pred):
    metrics = {}
    for i, t in enumerate(cfg.TARGETS):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        metrics[t] = {
            "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
            "MAE":  float(mean_absolute_error(yt, yp)),
            "R2":   float(r2_score(yt, yp)),
            "MAPE": float(np.mean(np.abs((yt - yp) / (np.abs(yt) + 1e-8))) * 100)
        }
    metrics["AVG"] = {
        k: float(np.mean([metrics[t][k] for t in cfg.TARGETS]))
        for k in ["RMSE", "MAE", "R2", "MAPE"]
    }
    return metrics

# ============================================================
# 4. SCALING HELPERS
# ============================================================
def fit_scalers(X_train, y_train):
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    X_s = scaler_X.fit_transform(X_train)
    y_s = scaler_y.fit_transform(y_train)
    return X_s, y_s, scaler_X, scaler_y

def transform_X(scaler_X, X):
    return scaler_X.transform(X)

def inverse_y(scaler_y, y_scaled):
    return scaler_y.inverse_transform(y_scaled)

# ============================================================
# 5. MODELS
# ============================================================
def train_rf(X, y):
    base = RandomForestRegressor(
        n_estimators=350,
        max_depth=18,
        min_samples_leaf=2,
        random_state=cfg.RANDOM_STATE,
        n_jobs=-1
    )
    model = MultiOutputRegressor(base, n_jobs=-1)
    model.fit(X, y)
    return model

def train_xgb(X, y):
    base = xgb.XGBRegressor(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=cfg.RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    )
    model = MultiOutputRegressor(base, n_jobs=-1)
    model.fit(X, y)
    return model

def create_dnn(input_dim):
    model = Sequential([
        Dense(256, activation="relu", input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.30),
        Dense(128, activation="relu"),
        BatchNormalization(),
        Dropout(0.25),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.20),
        Dense(32, activation="relu"),
        Dense(len(cfg.TARGETS))
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
    return model

def get_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=cfg.DNN_PATIENCE, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6, verbose=0)
    ]

def predict_model(model, X_scaled, scaler_y, is_dnn=False):
    if is_dnn:
        pred_s = model.predict(X_scaled, verbose=0)
    else:
        pred_s = model.predict(X_scaled)
    return inverse_y(scaler_y, pred_s)

# ============================================================
# 6. EXPERIMENT A – Train only on Current
# ============================================================
def run_experiment_A(X_cur, y_cur, X_50, y_50, X_80, y_80):
    print("\n" + "="*65)
    print("EXPERIMENT A: Train on Current only → Test on 2050 & 2080")
    print("="*65)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_cur, y_cur, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE
    )

    X_tr_s, y_tr_s, scaler_X, scaler_y = fit_scalers(X_tr, y_tr)
    X_te_s = transform_X(scaler_X, X_te)
    X_50_s = transform_X(scaler_X, X_50)
    X_80_s = transform_X(scaler_X, X_80)

    results = {}

    # Random Forest
    print("→ Training Random Forest ...")
    rf = train_rf(X_tr_s, y_tr_s)
    results["RF"] = {
        "current": calc_metrics(y_te, predict_model(rf, X_te_s, scaler_y)),
        "2050":    calc_metrics(y_50, predict_model(rf, X_50_s, scaler_y)),
        "2080":    calc_metrics(y_80, predict_model(rf, X_80_s, scaler_y)),
        "model": rf, "scaler_X": scaler_X, "scaler_y": scaler_y
    }

    # XGBoost
    print("→ Training XGBoost ...")
    xgb_model = train_xgb(X_tr_s, y_tr_s)
    results["XGB"] = {
        "current": calc_metrics(y_te, predict_model(xgb_model, X_te_s, scaler_y)),
        "2050":    calc_metrics(y_50, predict_model(xgb_model, X_50_s, scaler_y)),
        "2080":    calc_metrics(y_80, predict_model(xgb_model, X_80_s, scaler_y)),
        "model": xgb_model, "scaler_X": scaler_X, "scaler_y": scaler_y
    }

    # DNN
    print("→ Training DNN ...")
    dnn = create_dnn(X_tr_s.shape[1])
    dnn.fit(X_tr_s, y_tr_s,
            validation_split=0.15,
            epochs=cfg.DNN_EPOCHS,
            batch_size=cfg.DNN_BATCH,
            callbacks=get_callbacks(),
            verbose=0)
    results["DNN"] = {
        "current": calc_metrics(y_te, predict_model(dnn, X_te_s, scaler_y, is_dnn=True)),
        "2050":    calc_metrics(y_50, predict_model(dnn, X_50_s, scaler_y, is_dnn=True)),
        "2080":    calc_metrics(y_80, predict_model(dnn, X_80_s, scaler_y, is_dnn=True)),
        "model": dnn, "scaler_X": scaler_X, "scaler_y": scaler_y
    }

    return results

# ============================================================
# 7. EXPERIMENT B – Transfer Learning
# ============================================================
def run_experiment_B(results_A, X_50, y_50, X_80, y_80):
    print("\n" + "="*65)
    print("EXPERIMENT B: Transfer Learning (limited future data)")
    print("="*65)

    results_B = {}

    for year, X_fut, y_fut in [("2050", X_50, y_50), ("2080", X_80, y_80)]:
        print(f"\n--- {year} ---")
        n_tl = max(30, int(len(X_fut) * cfg.TL_FRACTION))
        idx = np.random.choice(len(X_fut), size=n_tl, replace=False)
        mask = np.ones(len(X_fut), dtype=bool)
        mask[idx] = False

        X_tl, y_tl = X_fut[idx], y_fut[idx]
        X_test, y_test = X_fut[mask], y_fut[mask]

        year_res = {}

        for name in ["RF", "XGB", "DNN"]:
            base_info = results_A[name]
            scaler_X = base_info["scaler_X"]
            scaler_y = base_info["scaler_y"]

            X_tl_s = transform_X(scaler_X, X_tl)
            X_test_s = transform_X(scaler_X, X_test)
            y_tl_s = scaler_y.transform(y_tl)

            if name == "DNN":
                # واقعی Transfer Learning
                dnn_tl = clone_model(base_info["model"])
                dnn_tl.set_weights(base_info["model"].get_weights())

                # freeze لایه‌های اولیه
                for layer in dnn_tl.layers[:3]:
                    layer.trainable = False

                dnn_tl.compile(optimizer=Adam(learning_rate=cfg.TL_LR), loss="mse")
                dnn_tl.fit(X_tl_s, y_tl_s,
                           validation_split=0.15,
                           epochs=cfg.TL_EPOCHS,
                           batch_size=cfg.DNN_BATCH,
                           callbacks=get_callbacks(),
                           verbose=0)
                pred = predict_model(dnn_tl, X_test_s, scaler_y, is_dnn=True)
                year_res[name] = calc_metrics(y_test, pred)
            else:
                # برای مدل‌های درختی: آموزش مجدد روی داده TL (adaptation)
                if name == "RF":
                    model_tl = train_rf(X_tl_s, y_tl_s)
                else:
                    model_tl = train_xgb(X_tl_s, y_tl_s)
                pred = predict_model(model_tl, X_test_s, scaler_y)
                year_res[name] = calc_metrics(y_test, pred)

        results_B[year] = year_res

    return results_B

# ============================================================
# 8. EXPERIMENT C – Full training on Future
# ============================================================
def run_experiment_C(X_50, y_50, X_80, y_80):
    print("\n" + "="*65)
    print("EXPERIMENT C: Full training from scratch on Future data")
    print("="*65)

    results_C = {}

    for year, X, y in [("2050", X_50, y_50), ("2080", X_80, y_80)]:
        print(f"\n--- Full training on {year} ---")
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE
        )

        X_tr_s, y_tr_s, scaler_X, scaler_y = fit_scalers(X_tr, y_tr)
        X_te_s = transform_X(scaler_X, X_te)

        year_res = {}

        print("  RF ...")
        rf = train_rf(X_tr_s, y_tr_s)
        year_res["RF"] = calc_metrics(y_te, predict_model(rf, X_te_s, scaler_y))

        print("  XGB ...")
        xgb_m = train_xgb(X_tr_s, y_tr_s)
        year_res["XGB"] = calc_metrics(y_te, predict_model(xgb_m, X_te_s, scaler_y))

        print("  DNN ...")
        dnn = create_dnn(X_tr_s.shape[1])
        dnn.fit(X_tr_s, y_tr_s,
                validation_split=0.15,
                epochs=cfg.DNN_EPOCHS,
                batch_size=cfg.DNN_BATCH,
                callbacks=get_callbacks(),
                verbose=0)
        year_res["DNN"] = calc_metrics(y_te, predict_model(dnn, X_te_s, scaler_y, is_dnn=True))

        results_C[year] = year_res

    return results_C

# ============================================================
# 9. DOMAIN SHIFT + SHAP
# ============================================================
def analyze_domain_shift(X_cur, X_50, X_80):
    print("\nAnalyzing Domain Shift (PCA)...")
    n = min(600, len(X_cur), len(X_50), len(X_80))
    all_X = np.vstack([X_cur[:n], X_50[:n], X_80[:n]])
    emb = PCA(n_components=2, random_state=cfg.RANDOM_STATE).fit_transform(
        StandardScaler().fit_transform(all_X)
    )
    labels = ["Current"]*n + ["2050"]*n + ["2080"]*n

    plt.figure(figsize=(9, 7))
    sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=labels, palette="Set1", alpha=0.65, s=28)
    plt.title("Domain Shift (PCA) – Current vs 2050 vs 2080", fontsize=13)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend(title="Climate")
    plt.tight_layout()
    path = cfg.OUTPUT_DIR / "domain_shift_pca.png"
    plt.savefig(path, dpi=220)
    plt.close()
    print(f"Saved: {path}")

def run_shap(results_A, X_sample, feature_names):
    print("\nRunning SHAP (first target of XGBoost)...")
    try:
        # برای سادگی روی اولین estimator از MultiOutput کار می‌کنیم
        model = results_A["XGB"]["model"].estimators_[0]
        scaler_X = results_A["XGB"]["scaler_X"]
        X_s = scaler_X.transform(X_sample[:120])

        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_s)

        mean_abs = np.abs(sv).mean(axis=0)
        importance = pd.DataFrame({
            "Feature": feature_names,
            "mean_abs_SHAP": mean_abs
        }).sort_values("mean_abs_SHAP", ascending=False)

        path = cfg.OUTPUT_DIR / "shap_importance_xgb_t1.csv"
        importance.to_csv(path, index=False)
        print(f"SHAP importance saved → {path}")
        print(importance.head(10).to_string(index=False))
    except Exception as e:
        print(f"SHAP skipped: {e}")

# ============================================================
# 10. SUMMARY TABLES
# ============================================================
def create_summary(res_A, res_B, res_C):
    rows = []

    for model_name in ["RF", "XGB", "DNN"]:
        for period in ["current", "2050", "2080"]:
            m = res_A[model_name][period]["AVG"]
            rows.append({
                "Experiment": "A_CurrentOnly",
                "Model": model_name,
                "Period": period,
                "RMSE": round(m["RMSE"], 2),
                "MAE":  round(m["MAE"], 2),
                "R2":   round(m["R2"], 4),
                "MAPE": round(m["MAPE"], 2)
            })

    for year in ["2050", "2080"]:
        for model_name in ["RF", "XGB", "DNN"]:
            m = res_B[year][model_name]["AVG"]
            rows.append({
                "Experiment": "B_TransferLearning",
                "Model": model_name,
                "Period": year,
                "RMSE": round(m["RMSE"], 2),
                "MAE":  round(m["MAE"], 2),
                "R2":   round(m["R2"], 4),
                "MAPE": round(m["MAPE"], 2)
            })

    for year in ["2050", "2080"]:
        for model_name in ["RF", "XGB", "DNN"]:
            m = res_C[year][model_name]["AVG"]
            rows.append({
                "Experiment": "C_FullFuture",
                "Model": model_name,
                "Period": year,
                "RMSE": round(m["RMSE"], 2),
                "MAE":  round(m["MAE"], 2),
                "R2":   round(m["R2"], 4),
                "MAPE": round(m["MAPE"], 2)
            })

    df = pd.DataFrame(rows)
    path = cfg.OUTPUT_DIR / "full_comparison.csv"
    df.to_csv(path, index=False)
    print("\n" + "="*90)
    print("FULL COMPARISON TABLE")
    print("="*90)
    print(df.to_string(index=False))
    print(f"\nSaved → {path}")
    return df

def print_degradation(res_A):
    print("\n" + "="*65)
    print("PERFORMANCE DEGRADATION (Experiment A)")
    print("="*65)
    for model in ["RF", "XGB", "DNN"]:
        r2_c = res_A[model]["current"]["AVG"]["R2"]
        r2_50 = res_A[model]["2050"]["AVG"]["R2"]
        r2_80 = res_A[model]["2080"]["AVG"]["R2"]
        print(f"{model:5s} | Current R²={r2_c:7.4f} → 2050 R²={r2_50:7.4f} (Δ={r2_50-r2_c:+.4f}) "
              f"→ 2080 R²={r2_80:7.4f} (Δ={r2_80-r2_c:+.4f})")

# ============================================================
# 11. MAIN
# ============================================================
def main():
    print("BuildingClimateML – Corrected Pipeline")
    print(f"Output: {cfg.OUTPUT_DIR}")

    df_cur, df_50, df_80 = load_data()
    df_50 = align_columns(df_cur, df_50)
    df_80 = align_columns(df_cur, df_80)

    X_cur, y_cur, feature_names = prepare_xy(df_cur)
    X_50,  y_50,  _ = prepare_xy(df_50, feature_names)
    X_80,  y_80,  _ = prepare_xy(df_80, feature_names)

    print(f"Features: {len(feature_names)} | Current: {len(X_cur)} | 2050: {len(X_50)} | 2080: {len(X_80)}")

    # Run experiments
    res_A = run_experiment_A(X_cur, y_cur, X_50, y_50, X_80, y_80)
    res_B = run_experiment_B(res_A, X_50, y_50, X_80, y_80)
    res_C = run_experiment_C(X_50, y_50, X_80, y_80)

    # Analyses
    analyze_domain_shift(X_cur, X_50, X_80)
    run_shap(res_A, X_cur, feature_names)

    # Reports
    create_summary(res_A, res_B, res_C)
    print_degradation(res_A)

    print("\n" + "="*65)
    print("Pipeline finished successfully!")
    print(f"All results saved to: {cfg.OUTPUT_DIR}")
    print("="*65)

if __name__ == "__main__":
    main()

BuildingClimateML – Corrected Pipeline
Output: D:\UPM\Machine Learning\Output_New_Pipeline
Loading real datasets...
Features: 107 | Current: 1826 | 2050: 909 | 2080: 991

EXPERIMENT A: Train on Current only → Test on 2050 & 2080
→ Training Random Forest ...
→ Training XGBoost ...
→ Training DNN ...

EXPERIMENT B: Transfer Learning (limited future data)

--- 2050 ---

--- 2080 ---

EXPERIMENT C: Full training from scratch on Future data

--- Full training on 2050 ---
  RF ...
  XGB ...
  DNN ...

--- Full training on 2080 ---
  RF ...
  XGB ...
  DNN ...

Analyzing Domain Shift (PCA)...
Saved: D:\UPM\Machine Learning\Output_New_Pipeline\domain_shift_pca.png

Running SHAP (first target of XGBoost)...
SHAP importance saved → D:\UPM\Machine Learning\Output_New_Pipeline\shap_importance_xgb_t1.csv
               Feature  mean_abs_SHAP
                    P9       0.373633
P25_WaterToAirHeatPump       0.210938
                    P3       0.153302
              P25_PTAC       0.115145
       

In [6]:
# ============================================================
# BuildingClimateML - Full Final Pipeline (Clean Version)
# ============================================================

import os
import warnings
import random
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition import PCA
from sklearn.utils import resample

import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

# ============================================================
# 1. CONFIG
# ============================================================
class Config:
    BASE_DIR = Path(r"D:\UPM\Machine Learning")
    CURRENT_FILE = BASE_DIR / "Scenario 1 Present Climate" / "Synthetic_data_sdv_dummies_1826_current climate.csv"
    FILE_2050 = BASE_DIR / "Scenario 2 2050" / "Synthetic_data_future_2050_processed.csv"
    FILE_2080 = BASE_DIR / "Scenario 2 2080" / "Synthetic_data_future_2080_processed.csv"
    OUTPUT_DIR = BASE_DIR / "Output_New_Pipeline"

    TARGETS = ["t1", "t2", "t3", "t4", "t5"]
    TEST_SIZE = 0.30
    RANDOM_STATE = 42
    TL_FRACTION = 0.30
    TL_EPOCHS = 50
    TL_LR = 1e-4
    DNN_EPOCHS = 120
    DNN_BATCH = 32
    DNN_PATIENCE = 15
    N_BOOTSTRAP = 30

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(cfg.RANDOM_STATE)

# ============================================================
# 2. DATA LOADING
# ============================================================
def load_data():
    print("Loading real datasets...")
    df_cur = pd.read_csv(cfg.CURRENT_FILE)
    df_50 = pd.read_csv(cfg.FILE_2050)
    df_80 = pd.read_csv(cfg.FILE_2080)
    return df_cur, df_50, df_80

def align_columns(df_ref, df_other):
    ref_feats = [c for c in df_ref.columns if c not in cfg.TARGETS]
    for c in ref_feats:
        if c not in df_other.columns:
            df_other[c] = 0.0
    return df_other[ref_feats + cfg.TARGETS]

def prepare_xy(df, feature_names=None):
    if feature_names is None:
        feature_names = [c for c in df.columns if c not in cfg.TARGETS]
    X = df[feature_names].values.astype(np.float64)
    y = df[cfg.TARGETS].values.astype(np.float64)
    return X, y, feature_names

# ============================================================
# 3. METRICS & SCALING
# ============================================================
def calc_metrics(y_true, y_pred):
    metrics = {}
    for i, t in enumerate(cfg.TARGETS):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        metrics[t] = {
            "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
            "MAE": float(mean_absolute_error(yt, yp)),
            "R2": float(r2_score(yt, yp)),
            "MAPE": float(np.mean(np.abs((yt - yp) / (np.abs(yt) + 1e-8))) * 100)
        }
    metrics["AVG"] = {
        k: float(np.mean([metrics[t][k] for t in cfg.TARGETS]))
        for k in ["RMSE", "MAE", "R2", "MAPE"]
    }
    return metrics

def fit_scalers(X_train, y_train):
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    X_scaled = scaler_X.fit_transform(X_train)
    y_scaled = scaler_y.fit_transform(y_train)
    return X_scaled, y_scaled, scaler_X, scaler_y

def transform_X(scaler_X, X):
    return scaler_X.transform(X)

def inverse_y(scaler_y, y_scaled):
    return scaler_y.inverse_transform(y_scaled)

# ============================================================
# 4. MODELS
# ============================================================
def train_rf(X, y):
    base = RandomForestRegressor(
        n_estimators=350,
        max_depth=18,
        min_samples_leaf=2,
        random_state=cfg.RANDOM_STATE,
        n_jobs=-1
    )
    model = MultiOutputRegressor(base, n_jobs=-1)
    model.fit(X, y)
    return model

def train_xgb(X, y):
    base = xgb.XGBRegressor(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=cfg.RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    )
    model = MultiOutputRegressor(base, n_jobs=-1)
    model.fit(X, y)
    return model

def create_dnn(input_dim):
    model = Sequential([
        Dense(256, activation="relu", input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.30),
        Dense(128, activation="relu"),
        BatchNormalization(),
        Dropout(0.25),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.20),
        Dense(32, activation="relu"),
        Dense(len(cfg.TARGETS))
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
    return model

def get_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=cfg.DNN_PATIENCE, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6, verbose=0)
    ]

def predict_model(model, X_scaled, scaler_y, is_dnn=False):
    if is_dnn:
        pred_s = model.predict(X_scaled, verbose=0)
    else:
        pred_s = model.predict(X_scaled)
    return inverse_y(scaler_y, pred_s)

# ============================================================
# 5. EXPERIMENT A
# ============================================================
def run_experiment_A(X_cur, y_cur, X_50, y_50, X_80, y_80):
    print("\n" + "=" * 65)
    print("EXPERIMENT A: Train on Current only")
    print("=" * 65)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_cur, y_cur, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE
    )

    X_tr_s, y_tr_s, scaler_X, scaler_y = fit_scalers(X_tr, y_tr)
    X_te_s = transform_X(scaler_X, X_te)
    X_50_s = transform_X(scaler_X, X_50)
    X_80_s = transform_X(scaler_X, X_80)

    results = {}

    print("-> RF ...")
    rf = train_rf(X_tr_s, y_tr_s)
    results["RF"] = {
        "current": calc_metrics(y_te, predict_model(rf, X_te_s, scaler_y)),
        "2050": calc_metrics(y_50, predict_model(rf, X_50_s, scaler_y)),
        "2080": calc_metrics(y_80, predict_model(rf, X_80_s, scaler_y)),
        "model": rf,
        "scaler_X": scaler_X,
        "scaler_y": scaler_y
    }

    print("-> XGB ...")
    xgb_model = train_xgb(X_tr_s, y_tr_s)
    results["XGB"] = {
        "current": calc_metrics(y_te, predict_model(xgb_model, X_te_s, scaler_y)),
        "2050": calc_metrics(y_50, predict_model(xgb_model, X_50_s, scaler_y)),
        "2080": calc_metrics(y_80, predict_model(xgb_model, X_80_s, scaler_y)),
        "model": xgb_model,
        "scaler_X": scaler_X,
        "scaler_y": scaler_y
    }

    print("-> DNN ...")
    dnn = create_dnn(X_tr_s.shape[1])
    dnn.fit(
        X_tr_s, y_tr_s,
        validation_split=0.15,
        epochs=cfg.DNN_EPOCHS,
        batch_size=cfg.DNN_BATCH,
        callbacks=get_callbacks(),
        verbose=0
    )
    results["DNN"] = {
        "current": calc_metrics(y_te, predict_model(dnn, X_te_s, scaler_y, is_dnn=True)),
        "2050": calc_metrics(y_50, predict_model(dnn, X_50_s, scaler_y, is_dnn=True)),
        "2080": calc_metrics(y_80, predict_model(dnn, X_80_s, scaler_y, is_dnn=True)),
        "model": dnn,
        "scaler_X": scaler_X,
        "scaler_y": scaler_y
    }

    return results

# ============================================================
# 6. EXPERIMENT B
# ============================================================
def run_experiment_B(results_A, X_50, y_50, X_80, y_80):
    print("\n" + "=" * 65)
    print("EXPERIMENT B: Transfer Learning")
    print("=" * 65)

    results_B = {}

    for year, X_fut, y_fut in [("2050", X_50, y_50), ("2080", X_80, y_80)]:
        print(f"\n--- {year} ---")
        n_tl = max(40, int(len(X_fut) * cfg.TL_FRACTION))
        idx = np.random.choice(len(X_fut), size=n_tl, replace=False)
        mask = np.ones(len(X_fut), dtype=bool)
        mask[idx] = False

        X_tl = X_fut[idx]
        y_tl = y_fut[idx]
        X_test = X_fut[mask]
        y_test = y_fut[mask]

        year_res = {}

        for name in ["RF", "XGB", "DNN"]:
            info = results_A[name]
            scaler_X = info["scaler_X"]
            scaler_y = info["scaler_y"]

            X_tl_s = transform_X(scaler_X, X_tl)
            X_test_s = transform_X(scaler_X, X_test)
            y_tl_s = scaler_y.transform(y_tl)

            if name == "DNN":
                dnn_tl = clone_model(info["model"])
                dnn_tl.set_weights(info["model"].get_weights())
                for layer in dnn_tl.layers[:3]:
                    layer.trainable = False
                dnn_tl.compile(optimizer=Adam(learning_rate=cfg.TL_LR), loss="mse")
                dnn_tl.fit(
                    X_tl_s, y_tl_s,
                    validation_split=0.15,
                    epochs=cfg.TL_EPOCHS,
                    batch_size=cfg.DNN_BATCH,
                    callbacks=get_callbacks(),
                    verbose=0
                )
                pred = predict_model(dnn_tl, X_test_s, scaler_y, is_dnn=True)
            else:
                if name == "RF":
                    model_tl = train_rf(X_tl_s, y_tl_s)
                else:
                    model_tl = train_xgb(X_tl_s, y_tl_s)
                pred = predict_model(model_tl, X_test_s, scaler_y)

            year_res[name] = calc_metrics(y_test, pred)

        results_B[year] = year_res

    return results_B

# ============================================================
# 7. EXPERIMENT C
# ============================================================
def run_experiment_C(X_50, y_50, X_80, y_80):
    print("\n" + "=" * 65)
    print("EXPERIMENT C: Full training on Future")
    print("=" * 65)

    results_C = {}

    for year, X, y in [("2050", X_50, y_50), ("2080", X_80, y_80)]:
        print(f"\n--- {year} ---")
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE
        )
        X_tr_s, y_tr_s, scaler_X, scaler_y = fit_scalers(X_tr, y_tr)
        X_te_s = transform_X(scaler_X, X_te)

        year_res = {}

        print("  RF ...")
        rf = train_rf(X_tr_s, y_tr_s)
        year_res["RF"] = calc_metrics(y_te, predict_model(rf, X_te_s, scaler_y))

        print("  XGB ...")
        xgb_m = train_xgb(X_tr_s, y_tr_s)
        year_res["XGB"] = calc_metrics(y_te, predict_model(xgb_m, X_te_s, scaler_y))

        print("  DNN ...")
        dnn = create_dnn(X_tr_s.shape[1])
        dnn.fit(
            X_tr_s, y_tr_s,
            validation_split=0.15,
            epochs=cfg.DNN_EPOCHS,
            batch_size=cfg.DNN_BATCH,
            callbacks=get_callbacks(),
            verbose=0
        )
        year_res["DNN"] = calc_metrics(y_te, predict_model(dnn, X_te_s, scaler_y, is_dnn=True))

        results_C[year] = year_res

    return results_C

# ============================================================
# 8. RECOVERY RATE + TABLES
# ============================================================
def calculate_recovery_rate(res_A, res_B, res_C):
    rows = []
    for year in ["2050", "2080"]:
        for model in ["RF", "XGB", "DNN"]:
            r2_a = res_A[model][year]["AVG"]["R2"]
            r2_b = res_B[year][model]["AVG"]["R2"]
            r2_c = res_C[year][model]["AVG"]["R2"]
            gap = r2_c - r2_a
            if abs(gap) < 1e-6:
                recovery = 0.0
            else:
                recovery = (r2_b - r2_a) / gap * 100.0

            rows.append({
                "Year": year,
                "Model": model,
                "R2_A_CurrentOnly": round(r2_a, 4),
                "R2_B_TransferLearning": round(r2_b, 4),
                "R2_C_FullFuture": round(r2_c, 4),
                "Absolute_Gain_by_TL": round(r2_b - r2_a, 4),
                "Recovery_Rate_%": round(recovery, 1)
            })

    df = pd.DataFrame(rows)
    path = cfg.OUTPUT_DIR / "recovery_rate_table.csv"
    df.to_csv(path, index=False)

    print("\n" + "=" * 95)
    print("RECOVERY RATE TABLE")
    print("=" * 95)
    print(df.to_string(index=False))
    print(f"Saved --> {path}")
    return df

def create_paper_main_table(res_A, res_B, res_C):
    rows = []
    for model in ["RF", "XGB", "DNN"]:
        m = res_A[model]["current"]["AVG"]
        rows.append({
            "Model": model,
            "Strategy": "Train on Current",
            "Climate": "Current",
            "R2": round(m["R2"], 3),
            "RMSE": round(m["RMSE"], 1),
            "MAE": round(m["MAE"], 1),
            "MAPE": round(m["MAPE"], 1)
        })
        for year in ["2050", "2080"]:
            for strat, source in [
                ("Train on Current", res_A[model][year]),
                ("Transfer Learning", res_B[year][model]),
                ("Full Future Training", res_C[year][model])
            ]:
                m = source["AVG"]
                rows.append({
                    "Model": model,
                    "Strategy": strat,
                    "Climate": year,
                    "R2": round(m["R2"], 3),
                    "RMSE": round(m["RMSE"], 1),
                    "MAE": round(m["MAE"], 1),
                    "MAPE": round(m["MAPE"], 1)
                })

    df = pd.DataFrame(rows)
    path = cfg.OUTPUT_DIR / "paper_main_results_table.csv"
    df.to_csv(path, index=False)
    print(f"Main paper table saved --> {path}")
    return df

# ============================================================
# 9. PLOTS
# ============================================================
def plot_performance_degradation(res_A):
    models = ["RF", "XGB", "DNN"]
    climates = ["current", "2050", "2080"]
    r2_data = {m: [res_A[m][c]["AVG"]["R2"] for c in climates] for m in models}

    x = np.arange(len(climates))
    width = 0.25
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

    fig, ax = plt.subplots(figsize=(9, 6))
    for i, model in enumerate(models):
        bars = ax.bar(x + i * width, r2_data[model], width, label=model, color=colors[i], edgecolor="black")
        for bar in bars:
            height = bar.get_height()
            ax.annotate(
                f"{height:.3f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=9
            )

    ax.set_ylabel("Average R2", fontsize=12)
    ax.set_xticks(x + width)
    ax.set_xticklabels(["Current", "2050", "2080"], fontsize=11)
    ax.set_ylim(0.55, 1.0)
    ax.legend(title="Model")
    ax.set_title("Performance Degradation under Climate Change\n(Models trained only on Current climate)", fontsize=13)
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()

    path = cfg.OUTPUT_DIR / "fig_performance_degradation.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")

def plot_recovery_comparison(res_A, res_B, res_C):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)

    for ax, year in zip(axes, ["2050", "2080"]):
        models = ["RF", "XGB", "DNN"]
        r2_a = [res_A[m][year]["AVG"]["R2"] for m in models]
        r2_b = [res_B[year][m]["AVG"]["R2"] for m in models]
        r2_c = [res_C[year][m]["AVG"]["R2"] for m in models]

        x = np.arange(len(models))
        width = 0.25

        ax.bar(x - width, r2_a, width, label="A: Current only", color="#d62728", edgecolor="black")
        ax.bar(x, r2_b, width, label="B: Transfer Learning", color="#2ca02c", edgecolor="black")
        ax.bar(x + width, r2_c, width, label="C: Full Future", color="#1f77b4", edgecolor="black")

        ax.set_title(f"Year {year}", fontsize=13)
        ax.set_xticks(x)
        ax.set_xticklabels(models, fontsize=11)
        ax.set_ylim(0.55, 0.95)
        ax.grid(axis="y", linestyle="--", alpha=0.6)

        if year == "2050":
            ax.set_ylabel("Average R2", fontsize=12)

    axes[1].legend(loc="lower right", fontsize=9)
    fig.suptitle("Comparison of Training Strategies under Future Climate", fontsize=14, y=1.02)
    plt.tight_layout()

    path = cfg.OUTPUT_DIR / "fig_strategy_comparison.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")

def plot_recovery_rate(df_recovery):
    fig, ax = plt.subplots(figsize=(9, 5.5))

    years = ["2050", "2080"]
    models = ["RF", "XGB", "DNN"]
    x = np.arange(len(models))
    width = 0.35

    for i, year in enumerate(years):
        vals = df_recovery[df_recovery["Year"] == year]["Recovery_Rate_%"].values
        bars = ax.bar(x + i * width, vals, width, label=year, edgecolor="black")
        for bar in bars:
            height = bar.get_height()
            ax.annotate(
                f"{height:.0f}%",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=9
            )

    ax.set_ylabel("Recovery Rate (%)", fontsize=12)
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(models, fontsize=11)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.legend(title="Year")
    ax.set_title("How much performance gap is recovered by Transfer Learning?", fontsize=13)
    ax.grid(axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()

    path = cfg.OUTPUT_DIR / "fig_recovery_rate.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")

# ============================================================
# 10. BOOTSTRAP
# ============================================================
def bootstrap_r2(model, X, y, scaler_X, scaler_y, is_dnn=False, n_boot=30):
    r2_list = []
    n = len(X)
    X_s = scaler_X.transform(X)

    for i in range(n_boot):
        idx = resample(np.arange(n), random_state=cfg.RANDOM_STATE + i)
        Xb = X_s[idx]
        yb = y[idx]

        if is_dnn:
            pred_s = model.predict(Xb, verbose=0)
        else:
            pred_s = model.predict(Xb)

        pred = scaler_y.inverse_transform(pred_s)
        r2s = [r2_score(yb[:, j], pred[:, j]) for j in range(y.shape[1])]
        r2_list.append(np.mean(r2s))

    return {
        "mean": float(np.mean(r2_list)),
        "ci_low": float(np.percentile(r2_list, 2.5)),
        "ci_high": float(np.percentile(r2_list, 97.5))
    }

def run_bootstrap(res_A, X_cur, y_cur, X_50, y_50, X_80, y_80):
    print("\nRunning Bootstrap CI for Experiment A ...")
    rows = []

    for model_name in ["RF", "XGB", "DNN"]:
        info = res_A[model_name]
        is_dnn = (model_name == "DNN")

        for period, X, y in [
            ("current", X_cur, y_cur),
            ("2050", X_50, y_50),
            ("2080", X_80, y_80)
        ]:
            n_sample = min(350, len(X))
            idx = np.random.choice(len(X), n_sample, replace=False)

            ci = bootstrap_r2(
                info["model"],
                X[idx],
                y[idx],
                info["scaler_X"],
                info["scaler_y"],
                is_dnn=is_dnn,
                n_boot=cfg.N_BOOTSTRAP
            )

            rows.append({
                "Model": model_name,
                "Period": period,
                "R2_mean": round(ci["mean"], 4),
                "R2_CI_low": round(ci["ci_low"], 4),
                "R2_CI_high": round(ci["ci_high"], 4)
            })

            print(f"  {model_name:5s} {period:8s}: {ci['mean']:.4f} [{ci['ci_low']:.4f} - {ci['ci_high']:.4f}]")

    df = pd.DataFrame(rows)
    path = cfg.OUTPUT_DIR / "bootstrap_r2_ci.csv"
    df.to_csv(path, index=False)
    print(f"Bootstrap CI saved --> {path}")
    return df

# ============================================================
# 11. DOMAIN SHIFT
# ============================================================
def analyze_domain_shift(X_cur, X_50, X_80):
    print("\nAnalyzing Domain Shift (PCA)...")
    n = min(500, len(X_cur), len(X_50), len(X_80))
    all_X = np.vstack([X_cur[:n], X_50[:n], X_80[:n]])
    emb = PCA(n_components=2, random_state=cfg.RANDOM_STATE).fit_transform(
        StandardScaler().fit_transform(all_X)
    )
    labels = ["Current"] * n + ["2050"] * n + ["2080"] * n

    plt.figure(figsize=(9, 7))
    sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=labels, palette="Set1", alpha=0.65, s=28)
    plt.title("Domain Shift (PCA) - Current vs 2050 vs 2080")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.tight_layout()

    path = cfg.OUTPUT_DIR / "domain_shift_pca.png"
    plt.savefig(path, dpi=220)
    plt.close()
    print(f"Saved: {path}")

# ============================================================
# 12. MAIN
# ============================================================
def main():
    print("BuildingClimateML - Final Complete Pipeline")
    print(f"Output directory: {cfg.OUTPUT_DIR}")

    df_cur, df_50, df_80 = load_data()
    df_50 = align_columns(df_cur, df_50)
    df_80 = align_columns(df_cur, df_80)

    X_cur, y_cur, feature_names = prepare_xy(df_cur)
    X_50, y_50, _ = prepare_xy(df_50, feature_names)
    X_80, y_80, _ = prepare_xy(df_80, feature_names)

    print(f"Features: {len(feature_names)} | Current: {len(X_cur)} | 2050: {len(X_50)} | 2080: {len(X_80)}")

    # Experiments
    res_A = run_experiment_A(X_cur, y_cur, X_50, y_50, X_80, y_80)
    res_B = run_experiment_B(res_A, X_50, y_50, X_80, y_80)
    res_C = run_experiment_C(X_50, y_50, X_80, y_80)

    # Domain Shift
    analyze_domain_shift(X_cur, X_50, X_80)

    # Tables
    df_recovery = calculate_recovery_rate(res_A, res_B, res_C)
    create_paper_main_table(res_A, res_B, res_C)

    # Plots
    plot_performance_degradation(res_A)
    plot_recovery_comparison(res_A, res_B, res_C)
    plot_recovery_rate(df_recovery)

    # Bootstrap
    run_bootstrap(res_A, X_cur, y_cur, X_50, y_50, X_80, y_80)

    print("\n" + "=" * 65)
    print("Pipeline finished successfully!")
    print(f"All results and figures saved to: {cfg.OUTPUT_DIR}")
    print("=" * 65)

if __name__ == "__main__":
    main()

BuildingClimateML - Final Complete Pipeline
Output directory: D:\UPM\Machine Learning\Output_New_Pipeline
Loading real datasets...
Features: 107 | Current: 1826 | 2050: 909 | 2080: 991

EXPERIMENT A: Train on Current only
-> RF ...
-> XGB ...
-> DNN ...

EXPERIMENT B: Transfer Learning

--- 2050 ---

--- 2080 ---

EXPERIMENT C: Full training on Future

--- 2050 ---
  RF ...
  XGB ...
  DNN ...

--- 2080 ---
  RF ...
  XGB ...
  DNN ...

Analyzing Domain Shift (PCA)...
Saved: D:\UPM\Machine Learning\Output_New_Pipeline\domain_shift_pca.png

RECOVERY RATE TABLE
Year Model  R2_A_CurrentOnly  R2_B_TransferLearning  R2_C_FullFuture  Absolute_Gain_by_TL  Recovery_Rate_%
2050    RF            0.8007                 0.7567           0.8549              -0.0439            -81.0
2050   XGB            0.8217                 0.8108           0.8849              -0.0109            -17.2
2050   DNN            0.7722                 0.8496           0.7735               0.0774           5802.7
2080  

In [1]:
# ============================================================
# BuildingClimateML
# RELIABILITY-AWARE CLIMATE ADAPTATION PIPELINE
#
# Research Questions
# ------------------------------------------------------------
# RQ1. How much does predictive reliability degrade when a
#     model trained under present climate is applied to
#     future climate conditions?
#
# RQ2. How is this degradation related to distribution/domain
#     shift between present and future conditions?
#
# RQ3. How much future data is required to recover predictive
#     reliability?
#
# RQ4. Does transfer learning provide an advantage over
#     training from scratch when future data are scarce?
#
# Models
# ------------------------------------------------------------
# 1. Random Forest
# 2. XGBoost
# 3. Deep Neural Network
#
# Experiments
# ------------------------------------------------------------
# A. Current-climate training -> future zero-shot evaluation
# B. Data-efficient future adaptation
# C. DNN transfer learning
# D. Future-climate training from scratch
# E. Domain-shift analysis
# F. Bootstrap uncertainty
# G. Repeated-sampling robustness
#
# IMPORTANT TERMINOLOGY
# ------------------------------------------------------------
# RF/XGB adaptation on future subsets is NOT called
# "Transfer Learning" because these tree models are retrained
# on the target-domain samples.
#
# DNN adaptation is genuine transfer learning because the
# present-climate model is initialized from the source-domain
# weights and fine-tuned on future-domain data.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    BatchNormalization,
    Input
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam

import shap


warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")


# ============================================================
# 1. CONFIGURATION
# ============================================================

class Config:

    # --------------------------------------------------------
    # Base directory
    # --------------------------------------------------------

    BASE_DIR = Path(r"D:\UPM\Machine Learning")

    CURRENT_FILE = (
        BASE_DIR
        / "Scenario 1 Present Climate"
        / "Synthetic_data_sdv_dummies_1826_current climate.csv"
    )

    FILE_2050 = (
        BASE_DIR
        / "Scenario 2 2050"
        / "Synthetic_data_future_2050_processed.csv"
    )

    FILE_2080 = (
        BASE_DIR
        / "Scenario 2 2080"
        / "Synthetic_data_future_2080_processed.csv"
    )

    OUTPUT_DIR = BASE_DIR / "Output_Reliability_Pipeline"


    # --------------------------------------------------------
    # Targets
    # --------------------------------------------------------

    TARGETS = [
        "t1",
        "t2",
        "t3",
        "t4",
        "t5"
    ]


    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    RANDOM_STATE = 42

    # Number of repeated experiments for each future-data
    # fraction
    N_REPEATS = 5

    # Seeds used for repeated future-data sampling
    REPEAT_SEEDS = [
        42,
        52,
        62,
        72,
        82
    ]


    # --------------------------------------------------------
    # Current-climate train/test split
    # --------------------------------------------------------

    CURRENT_TEST_SIZE = 0.30


    # --------------------------------------------------------
    # Fixed future test set
    #
    # IMPORTANT:
    # All future-data adaptation experiments use exactly the
    # same fixed test set.
    # --------------------------------------------------------

    FUTURE_TEST_SIZE = 0.20


    # --------------------------------------------------------
    # Future-data fractions
    #
    # These fractions refer to the FUTURE TRAINING POOL,
    # not the complete future dataset.
    # --------------------------------------------------------

    FUTURE_FRACTIONS = [
        0.00,
        0.05,
        0.10,
        0.20,
        0.30,
        0.50,
        0.75,
        1.00
    ]


    # --------------------------------------------------------
    # Models
    # --------------------------------------------------------

    RF_N_ESTIMATORS = 350
    RF_MAX_DEPTH = 18
    RF_MIN_SAMPLES_LEAF = 2


    XGB_N_ESTIMATORS = 400
    XGB_MAX_DEPTH = 6
    XGB_LEARNING_RATE = 0.05
    XGB_SUBSAMPLE = 0.85
    XGB_COLSAMPLE = 0.85
    XGB_REG_LAMBDA = 1.0


    # --------------------------------------------------------
    # DNN
    # --------------------------------------------------------

    DNN_EPOCHS = 150
    DNN_BATCH = 32
    DNN_PATIENCE = 15

    DNN_LR = 0.001

    # Transfer-learning learning rate
    TL_LR = 1e-4

    # Transfer-learning epochs
    TL_EPOCHS = 80

    # Fraction of adaptation data used internally for validation
    DNN_VALIDATION_SIZE = 0.15


    # --------------------------------------------------------
    # DNN freezing strategies
    #
    # 0 = fine-tune all layers
    # 1 = freeze first Dense+BN block
    # 2 = freeze first two Dense+BN blocks
    # --------------------------------------------------------

    DNN_FREEZE_STRATEGIES = [
        0,
        1,
        2
    ]


    # --------------------------------------------------------
    # Bootstrap
    # --------------------------------------------------------

    BOOTSTRAP_ITERATIONS = 1000
    CONFIDENCE_LEVEL = 0.95


    # --------------------------------------------------------
    # Domain shift
    # --------------------------------------------------------

    DOMAIN_SHIFT_SAMPLE = 600

    # Number of permutations/repeated MMD estimates
    MMD_REPEATS = 5


    # --------------------------------------------------------
    # SHAP
    # --------------------------------------------------------

    SHAP_SAMPLE_SIZE = 200


cfg = Config()

cfg.OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_global_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    tf.random.set_seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


set_global_seed(cfg.RANDOM_STATE)


# ============================================================
# 3. UTILITY FUNCTIONS
# ============================================================

def save_json(data, path):

    def convert(obj):

        if isinstance(obj, np.integer):
            return int(obj)

        if isinstance(obj, np.floating):
            return float(obj)

        if isinstance(obj, np.ndarray):
            return obj.tolist()

        return obj

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            indent=4,
            default=convert
        )


def ensure_directory(path):

    Path(path).mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 4. DATA LOADING
# ============================================================

def load_dataset(path, name):

    path = Path(path)

    if not path.exists():

        raise FileNotFoundError(
            f"\n{name} dataset not found:\n{path}\n"
            f"Please update the path in Config."
        )

    df = pd.read_csv(path)

    print(
        f"{name}: "
        f"{df.shape[0]} rows × {df.shape[1]} columns"
    )

    return df


def validate_targets(df, dataset_name):

    missing = [
        t for t in cfg.TARGETS
        if t not in df.columns
    ]

    if missing:

        raise ValueError(
            f"{dataset_name} is missing targets: {missing}"
        )


def validate_numeric(df, dataset_name):

    non_numeric = [
        c for c in df.columns
        if not pd.api.types.is_numeric_dtype(df[c])
    ]

    if non_numeric:

        raise ValueError(
            f"{dataset_name} contains non-numeric columns:\n"
            f"{non_numeric}"
        )


# ============================================================
# 5. FEATURE SCHEMA
# ============================================================

def get_feature_names(df):

    return [
        c for c in df.columns
        if c not in cfg.TARGETS
    ]


def align_future_schema(
    df_current,
    df_future,
    future_name
):

    current_features = get_feature_names(
        df_current
    )

    future_features = get_feature_names(
        df_future
    )

    missing = [
        c for c in current_features
        if c not in future_features
    ]

    extra = [
        c for c in future_features
        if c not in current_features
    ]

    # --------------------------------------------------------
    # Missing feature columns are NOT silently filled with zero.
    # --------------------------------------------------------

    if missing:

        raise ValueError(
            f"\n{future_name} is missing feature columns.\n"
            f"This must be resolved before modeling.\n\n"
            f"Missing columns ({len(missing)}):\n"
            f"{missing}"
        )

    # Extra columns are ignored deliberately because the
    # current-climate feature schema defines the source model.
    if extra:

        print(
            f"\nWARNING: {future_name} contains "
            f"{len(extra)} extra feature columns."
        )

        print(
            "Extra columns will be ignored."
        )

    ordered_columns = (
        current_features
        + cfg.TARGETS
    )

    return df_future[ordered_columns].copy()


# ============================================================
# 6. PREPARE X AND Y
# ============================================================

def prepare_xy(
    df,
    feature_names
):

    X = (
        df[feature_names]
        .values
        .astype(np.float64)
    )

    y = (
        df[cfg.TARGETS]
        .values
        .astype(np.float64)
    )

    return X, y


# ============================================================
# 7. SCALING
# ============================================================

def fit_scalers(
    X_train,
    y_train
):

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_scaled = scaler_X.fit_transform(
        X_train
    )

    y_scaled = scaler_y.fit_transform(
        y_train
    )

    return (
        X_scaled,
        y_scaled,
        scaler_X,
        scaler_y
    )


def transform_X(
    scaler_X,
    X
):

    return scaler_X.transform(X)


def transform_y(
    scaler_y,
    y
):

    return scaler_y.transform(y)


def inverse_y(
    scaler_y,
    y_scaled
):

    return scaler_y.inverse_transform(
        y_scaled
    )


# ============================================================
# 8. METRICS
# ============================================================

def safe_r2(
    y_true,
    y_pred
):

    try:

        return float(
            r2_score(
                y_true,
                y_pred
            )
        )

    except Exception:

        return np.nan


def calculate_metrics(
    y_true,
    y_pred
):

    results = {}

    for i, target in enumerate(
        cfg.TARGETS
    ):

        yt = y_true[:, i]
        yp = y_pred[:, i]

        rmse = np.sqrt(
            mean_squared_error(
                yt,
                yp
            )
        )

        mae = mean_absolute_error(
            yt,
            yp
        )

        r2 = safe_r2(
            yt,
            yp
        )

        results[target] = {

            "RMSE": float(rmse),

            "MAE": float(mae),

            "R2": float(r2)
        }

    return results


# ============================================================
# 9. BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

def bootstrap_metric_ci(
    y_true,
    y_pred,
    metric,
    n_bootstrap=1000,
    seed=42,
    confidence=0.95
):

    rng = np.random.default_rng(
        seed
    )

    n = len(y_true)

    values = []

    for _ in range(
        n_bootstrap
    ):

        indices = rng.integers(
            0,
            n,
            size=n
        )

        yt = y_true[indices]
        yp = y_pred[indices]

        if metric == "R2":

            value = r2_score(
                yt,
                yp
            )

        elif metric == "RMSE":

            value = np.sqrt(
                mean_squared_error(
                    yt,
                    yp
                )
            )

        elif metric == "MAE":

            value = mean_absolute_error(
                yt,
                yp
            )

        else:

            raise ValueError(
                f"Unknown metric: {metric}"
            )

        values.append(
            value
        )

    alpha = 1 - confidence

    lower = np.percentile(
        values,
        100 * alpha / 2
    )

    upper = np.percentile(
        values,
        100 * (1 - alpha / 2)
    )

    return {
        "estimate": float(
            np.mean(values)
        ),
        "lower": float(lower),
        "upper": float(upper)
    }


def calculate_bootstrap_metrics(
    y_true,
    y_pred,
    seed=42
):

    output = {}

    for i, target in enumerate(
        cfg.TARGETS
    ):

        output[target] = {}

        yt = y_true[:, i]
        yp = y_pred[:, i]

        for metric in [
            "R2",
            "RMSE",
            "MAE"
        ]:

            output[target][metric] = (
                bootstrap_metric_ci(
                    yt,
                    yp,
                    metric,
                    n_bootstrap=cfg.BOOTSTRAP_ITERATIONS,
                    seed=seed,
                    confidence=cfg.CONFIDENCE_LEVEL
                )
            )

    return output


# ============================================================
# 10. MODELS
# ============================================================

def train_random_forest(
    X,
    y,
    seed
):

    model = RandomForestRegressor(

        n_estimators=cfg.RF_N_ESTIMATORS,

        max_depth=cfg.RF_MAX_DEPTH,

        min_samples_leaf=cfg.RF_MIN_SAMPLES_LEAF,

        random_state=seed,

        n_jobs=-1
    )

    model = MultiOutputRegressor(
        model,
        n_jobs=-1
    )

    model.fit(
        X,
        y
    )

    return model


def train_xgboost(
    X,
    y,
    seed
):

    base = xgb.XGBRegressor(

        n_estimators=cfg.XGB_N_ESTIMATORS,

        max_depth=cfg.XGB_MAX_DEPTH,

        learning_rate=cfg.XGB_LEARNING_RATE,

        subsample=cfg.XGB_SUBSAMPLE,

        colsample_bytree=cfg.XGB_COLSAMPLE,

        reg_lambda=cfg.XGB_REG_LAMBDA,

        objective="reg:squarederror",

        random_state=seed,

        n_jobs=-1,

        verbosity=0
    )

    model = MultiOutputRegressor(
        base,
        n_jobs=-1
    )

    model.fit(
        X,
        y
    )

    return model


# ============================================================
# 11. DNN
# ============================================================

def create_dnn(
    input_dim,
    seed=42
):

    set_global_seed(seed)

    model = tf.keras.Sequential(
        [

            Input(
                shape=(input_dim,)
            ),

            Dense(
                256,
                activation="relu"
            ),

            BatchNormalization(),

            Dropout(
                0.30
            ),

            Dense(
                128,
                activation="relu"
            ),

            BatchNormalization(),

            Dropout(
                0.25
            ),

            Dense(
                64,
                activation="relu"
            ),

            BatchNormalization(),

            Dropout(
                0.20
            ),

            Dense(
                32,
                activation="relu"
            ),

            Dense(
                len(cfg.TARGETS)
            )
        ]
    )

    model.compile(

        optimizer=Adam(
            learning_rate=cfg.DNN_LR
        ),

        loss="mse"
    )

    return model


def get_dnn_callbacks():

    return [

        EarlyStopping(

            monitor="val_loss",

            patience=cfg.DNN_PATIENCE,

            restore_best_weights=True,

            verbose=0
        ),

        ReduceLROnPlateau(

            monitor="val_loss",

            factor=0.5,

            patience=7,

            min_lr=1e-6,

            verbose=0
        )
    ]


def train_dnn(
    X,
    y,
    seed
):

    set_global_seed(seed)

    model = create_dnn(
        X.shape[1],
        seed
    )

    model.fit(

        X,
        y,

        validation_split=cfg.DNN_VALIDATION_SIZE,

        epochs=cfg.DNN_EPOCHS,

        batch_size=cfg.DNN_BATCH,

        callbacks=get_dnn_callbacks(),

        verbose=0
    )

    return model


# ============================================================
# 12. DNN TRANSFER LEARNING
# ============================================================

def clone_dnn_model(
    source_model
):

    model = tf.keras.models.clone_model(
        source_model
    )

    model.set_weights(
        source_model.get_weights()
    )

    return model


def apply_freezing_strategy(
    model,
    freeze_strategy
):

    # First make all layers trainable
    for layer in model.layers:

        layer.trainable = True


    if freeze_strategy == 0:

        return


    # Identify Dense/BN layers
    trainable_blocks = []

    dense_seen = 0

    for layer in model.layers:

        if isinstance(
            layer,
            Dense
        ):

            dense_seen += 1

            if dense_seen <= freeze_strategy:

                trainable_blocks.append(
                    layer
                )


    # Freeze corresponding early layers and their BN layers
    if freeze_strategy >= 1:

        count_dense = 0

        for layer in model.layers:

            if isinstance(
                layer,
                Dense
            ):

                count_dense += 1

                if count_dense <= freeze_strategy:

                    layer.trainable = False

            elif isinstance(
                layer,
                BatchNormalization
            ):

                # Freeze BN layers associated with the
                # early representation
                if count_dense <= freeze_strategy:

                    layer.trainable = False


def transfer_learn_dnn(
    source_model,
    X_future,
    y_future,
    X_test,
    scaler_y,
    freeze_strategy,
    seed
):

    set_global_seed(seed)

    model = clone_dnn_model(
        source_model
    )

    apply_freezing_strategy(
        model,
        freeze_strategy
    )

    model.compile(

        optimizer=Adam(
            learning_rate=cfg.TL_LR
        ),

        loss="mse"
    )

    model.fit(

        X_future,
        y_future,

        validation_split=cfg.DNN_VALIDATION_SIZE,

        epochs=cfg.TL_EPOCHS,

        batch_size=cfg.DNN_BATCH,

        callbacks=get_dnn_callbacks(),

        verbose=0
    )

    pred_scaled = model.predict(
        X_test,
        verbose=0
    )

    pred = inverse_y(
        scaler_y,
        pred_scaled
    )

    return model, pred


# ============================================================
# 13. FIXED FUTURE TEST SPLIT
# ============================================================

def create_fixed_future_split(
    X,
    y,
    seed=42
):

    X_pool, X_test, y_pool, y_test = (
        train_test_split(

            X,
            y,

            test_size=cfg.FUTURE_TEST_SIZE,

            random_state=seed
        )
    )

    return (
        X_pool,
        X_test,
        y_pool,
        y_test
    )


# ============================================================
# 14. CURRENT-CLIMATE SOURCE MODELS
# ============================================================

def train_current_models(
    X_current,
    y_current
):

    print(
        "\n"
        + "=" * 75
    )

    print(
        "SOURCE-DOMAIN TRAINING: PRESENT CLIMATE"
    )

    print(
        "=" * 75
    )


    X_train, X_test, y_train, y_test = (
        train_test_split(

            X_current,

            y_current,

            test_size=cfg.CURRENT_TEST_SIZE,

            random_state=cfg.RANDOM_STATE
        )
    )


    (
        X_train_s,
        y_train_s,
        scaler_X,
        scaler_y
    ) = fit_scalers(
        X_train,
        y_train
    )


    X_test_s = transform_X(
        scaler_X,
        X_test
    )


    models = {}

    predictions = {}

    # --------------------------------------------------------
    # RF
    # --------------------------------------------------------

    print(
        "Training Random Forest..."
    )

    rf = train_random_forest(
        X_train_s,
        y_train_s,
        cfg.RANDOM_STATE
    )

    pred_rf = inverse_y(
        scaler_y,
        rf.predict(X_test_s)
    )

    models["RF"] = rf

    predictions["RF"] = pred_rf


    # --------------------------------------------------------
    # XGB
    # --------------------------------------------------------

    print(
        "Training XGBoost..."
    )

    xgb_model = train_xgboost(
        X_train_s,
        y_train_s,
        cfg.RANDOM_STATE
    )

    pred_xgb = inverse_y(
        scaler_y,
        xgb_model.predict(X_test_s)
    )

    models["XGB"] = xgb_model

    predictions["XGB"] = pred_xgb


    # --------------------------------------------------------
    # DNN
    # --------------------------------------------------------

    print(
        "Training DNN..."
    )

    dnn = train_dnn(
        X_train_s,
        y_train_s,
        cfg.RANDOM_STATE
    )

    pred_dnn_s = dnn.predict(
        X_test_s,
        verbose=0
    )

    pred_dnn = inverse_y(
        scaler_y,
        pred_dnn_s
    )

    models["DNN"] = dnn

    predictions["DNN"] = pred_dnn


    return {

        "models": models,

        "predictions": predictions,

        "X_train": X_train,

        "X_test": X_test,

        "y_train": y_train,

        "y_test": y_test,

        "scaler_X": scaler_X,

        "scaler_y": scaler_y
    }


# ============================================================
# 15. ZERO-SHOT FUTURE EVALUATION
# ============================================================

def evaluate_zero_shot(
    current_results,
    X_future_test,
    y_future_test
):

    results = {}

    scaler_X = current_results[
        "scaler_X"
    ]

    scaler_y = current_results[
        "scaler_y"
    ]

    X_future_s = transform_X(
        scaler_X,
        X_future_test
    )


    for model_name in [
        "RF",
        "XGB",
        "DNN"
    ]:

        model = current_results[
            "models"
        ][model_name]


        if model_name == "DNN":

            pred_s = model.predict(
                X_future_s,
                verbose=0
            )

        else:

            pred_s = model.predict(
                X_future_s
            )


        pred = inverse_y(
            scaler_y,
            pred_s
        )

        results[model_name] = {

            "predictions": pred,

            "metrics": calculate_metrics(
                y_future_test,
                pred
            ),

            "bootstrap": calculate_bootstrap_metrics(
                y_future_test,
                pred,
                seed=cfg.RANDOM_STATE
            )
        }


    return results


# ============================================================
# 16. FUTURE-DATA SUBSET SAMPLING
# ============================================================

def sample_future_training_data(
    X_pool,
    y_pool,
    fraction,
    seed
):

    if fraction <= 0:

        return None, None


    if fraction >= 1.0:

        return (
            X_pool.copy(),
            y_pool.copy()
        )


    rng = np.random.default_rng(
        seed
    )

    n = max(
        1,
        int(
            len(X_pool) * fraction
        )
    )

    indices = rng.choice(

        len(X_pool),

        size=n,

        replace=False
    )

    return (
        X_pool[indices],
        y_pool[indices]
    )


# ============================================================
# 17. FUTURE ADAPTATION
#
# RF/XGB:
# Future-domain retraining
#
# DNN:
# True transfer learning
# ============================================================

def run_future_adaptation(
    current_results,
    X_future_pool,
    y_future_pool,
    X_future_test,
    y_future_test,
    fraction,
    seed,
    freeze_strategy=1
):

    output = {}


    # --------------------------------------------------------
    # 0% = Zero-shot
    # --------------------------------------------------------

    if fraction == 0:

        return evaluate_zero_shot(
            current_results,
            X_future_test,
            y_future_test
        )


    # --------------------------------------------------------
    # Sample future adaptation data
    # --------------------------------------------------------

    X_adapt, y_adapt = (
        sample_future_training_data(

            X_future_pool,

            y_future_pool,

            fraction,

            seed
        )
    )


    # --------------------------------------------------------
    # Source scalers
    #
    # We intentionally keep source-climate scaling for
    # transfer learning so that the DNN receives data in the
    # source representation.
    # --------------------------------------------------------

    scaler_X_source = current_results[
        "scaler_X"
    ]

    scaler_y_source = current_results[
        "scaler_y"
    ]

    X_adapt_s_source = transform_X(
        scaler_X_source,
        X_adapt
    )

    X_test_s_source = transform_X(
        scaler_X_source,
        X_future_test
    )

    y_adapt_s_source = transform_y(
        scaler_y_source,
        y_adapt
    )


    # --------------------------------------------------------
    # RF: future-domain retraining
    # --------------------------------------------------------

    rf_scaler_X = StandardScaler()

    rf_scaler_y = StandardScaler()

    X_rf = rf_scaler_X.fit_transform(
        X_adapt
    )

    y_rf = rf_scaler_y.fit_transform(
        y_adapt
    )

    X_rf_test = rf_scaler_X.transform(
        X_future_test
    )

    rf = train_random_forest(
        X_rf,
        y_rf,
        seed
    )

    pred_rf = inverse_y(
        rf_scaler_y,
        rf.predict(X_rf_test)
    )


    output["RF"] = {

        "predictions": pred_rf,

        "metrics": calculate_metrics(
            y_future_test,
            pred_rf
        ),

        "bootstrap": calculate_bootstrap_metrics(
            y_future_test,
            pred_rf,
            seed=seed
        ),

        "method": "Future-data retraining"
    }


    # --------------------------------------------------------
    # XGBoost: future-domain retraining
    # --------------------------------------------------------

    xgb_scaler_X = StandardScaler()

    xgb_scaler_y = StandardScaler()

    X_xgb = xgb_scaler_X.fit_transform(
        X_adapt
    )

    y_xgb = xgb_scaler_y.fit_transform(
        y_adapt
    )

    X_xgb_test = xgb_scaler_X.transform(
        X_future_test
    )

    xgb_model = train_xgboost(
        X_xgb,
        y_xgb,
        seed
    )

    pred_xgb = inverse_y(
        xgb_scaler_y,
        xgb_model.predict(
            X_xgb_test
        )
    )


    output["XGB"] = {

        "predictions": pred_xgb,

        "metrics": calculate_metrics(
            y_future_test,
            pred_xgb
        ),

        "bootstrap": calculate_bootstrap_metrics(
            y_future_test,
            pred_xgb,
            seed=seed
        ),

        "method": "Future-data retraining"
    }


    # --------------------------------------------------------
    # DNN: TRUE TRANSFER LEARNING
    # --------------------------------------------------------

    source_dnn = current_results[
        "models"
    ]["DNN"]


    dnn_tl, pred_dnn = transfer_learn_dnn(

        source_dnn,

        X_adapt_s_source,

        y_adapt_s_source,

        X_test_s_source,

        scaler_y_source,

        freeze_strategy,

        seed
    )


    output["DNN"] = {

        "predictions": pred_dnn,

        "metrics": calculate_metrics(
            y_future_test,
            pred_dnn
        ),

        "bootstrap": calculate_bootstrap_metrics(
            y_future_test,
            pred_dnn,
            seed=seed
        ),

        "method": (
            "Transfer learning "
            f"(freeze strategy={freeze_strategy})"
        )
    }


    return output


# ============================================================
# 18. FUTURE TRAINING FROM SCRATCH
#
# Full future training is treated as the upper-performance
# reference.
# ============================================================

def train_full_future_models(
    X_future_pool,
    y_future_pool,
    X_future_test,
    y_future_test,
    seed
):

    output = {}


    # --------------------------------------------------------
    # RF
    # --------------------------------------------------------

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_s = scaler_X.fit_transform(
        X_future_pool
    )

    y_train_s = scaler_y.fit_transform(
        y_future_pool
    )

    X_test_s = scaler_X.transform(
        X_future_test
    )

    rf = train_random_forest(
        X_train_s,
        y_train_s,
        seed
    )

    pred_rf = inverse_y(
        scaler_y,
        rf.predict(
            X_test_s
        )
    )

    output["RF"] = {

        "predictions": pred_rf,

        "metrics": calculate_metrics(
            y_future_test,
            pred_rf
        ),

        "bootstrap": calculate_bootstrap_metrics(
            y_future_test,
            pred_rf,
            seed=seed
        )
    }


    # --------------------------------------------------------
    # XGB
    # --------------------------------------------------------

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_s = scaler_X.fit_transform(
        X_future_pool
    )

    y_train_s = scaler_y.fit_transform(
        y_future_pool
    )

    X_test_s = scaler_X.transform(
        X_future_test
    )

    xgb_model = train_xgboost(
        X_train_s,
        y_train_s,
        seed
    )

    pred_xgb = inverse_y(
        scaler_y,
        xgb_model.predict(
            X_test_s
        )
    )

    output["XGB"] = {

        "predictions": pred_xgb,

        "metrics": calculate_metrics(
            y_future_test,
            pred_xgb
        ),

        "bootstrap": calculate_bootstrap_metrics(
            y_future_test,
            pred_xgb,
            seed=seed
        )
    }


    # --------------------------------------------------------
    # DNN
    # --------------------------------------------------------

    dnn = train_dnn(
        X_train_s,
        y_train_s,
        seed
    )

    pred_dnn_s = dnn.predict(
        X_test_s,
        verbose=0
    )

    pred_dnn = inverse_y(
        scaler_y,
        pred_dnn_s
    )

    output["DNN"] = {

        "predictions": pred_dnn,

        "metrics": calculate_metrics(
            y_future_test,
            pred_dnn
        ),

        "bootstrap": calculate_bootstrap_metrics(
            y_future_test,
            pred_dnn,
            seed=seed
        )
    }


    return output


# ============================================================
# 19. RELIABILITY DEGRADATION
# ============================================================

def calculate_degradation(
    current_metrics,
    future_metrics
):

    rows = []

    for model in [
        "RF",
        "XGB",
        "DNN"
    ]:

        for target in cfg.TARGETS:

            current_r2 = (
                current_metrics[
                    model
                ][target]["R2"]
            )

            future_r2 = (
                future_metrics[
                    model
                ][target]["R2"]
            )

            rows.append({

                "Model": model,

                "Target": target,

                "Current_R2": current_r2,

                "Future_R2": future_r2,

                "R2_Degradation": (
                    future_r2
                    - current_r2
                ),

                "Absolute_R2_Loss": (
                    current_r2
                    - future_r2
                )
            })

    return pd.DataFrame(rows)


# ============================================================
# 20. RECOVERY RATE
#
# R2:
#
#   (Adapted - ZeroShot)
#   -------------------- × 100
#   (Full - ZeroShot)
#
# RMSE:
#
#   (ZeroShot - Adapted)
#   -------------------- × 100
#   (ZeroShot - Full)
# ============================================================

def calculate_recovery(
    zero_metrics,
    adapted_metrics,
    full_metrics
):

    rows = []

    for model in [
        "RF",
        "XGB",
        "DNN"
    ]:

        for target in cfg.TARGETS:

            zero_r2 = zero_metrics[
                model
            ][target]["R2"]

            adapted_r2 = adapted_metrics[
                model
            ][target]["R2"]

            full_r2 = full_metrics[
                model
            ][target]["R2"]


            denominator = (
                full_r2
                - zero_r2
            )

            if abs(denominator) < 1e-12:

                r2_recovery = np.nan

            else:

                r2_recovery = (
                    (
                        adapted_r2
                        - zero_r2
                    )
                    / denominator
                ) * 100


            zero_rmse = zero_metrics[
                model
            ][target]["RMSE"]

            adapted_rmse = adapted_metrics[
                model
            ][target]["RMSE"]

            full_rmse = full_metrics[
                model
            ][target]["RMSE"]


            denominator_rmse = (
                zero_rmse
                - full_rmse
            )

            if abs(denominator_rmse) < 1e-12:

                rmse_recovery = np.nan

            else:

                rmse_recovery = (
                    (
                        zero_rmse
                        - adapted_rmse
                    )
                    / denominator_rmse
                ) * 100


            rows.append({

                "Model": model,

                "Target": target,

                "R2_Recovery_Percent": (
                    r2_recovery
                ),

                "RMSE_Recovery_Percent": (
                    rmse_recovery
                )
            })


    return pd.DataFrame(rows)


# ============================================================
# 21. DOMAIN SHIFT
#
# PCA + MMD
# ============================================================

def rbf_kernel(
    X,
    Y,
    gamma=None
):

    distances = pairwise_distances(
        X,
        Y,
        metric="sqeuclidean"
    )

    if gamma is None:

        positive = distances[
            distances > 0
        ]

        if len(positive) == 0:

            gamma = 1.0

        else:

            median_distance = np.median(
                positive
            )

            gamma = (
                1.0
                / (
                    2
                    * median_distance
                )
            )

    return np.exp(
        -gamma * distances
    )


def calculate_mmd(
    X,
    Y
):

    Kxx = rbf_kernel(
        X,
        X
    )

    Kyy = rbf_kernel(
        Y,
        Y
    )

    Kxy = rbf_kernel(
        X,
        Y
    )

    n = len(X)
    m = len(Y)

    if n > 1:

        Kxx_mean = (
            (
                Kxx.sum()
                - np.trace(Kxx)
            )
            / (
                n * (n - 1)
            )
        )

    else:

        Kxx_mean = 0


    if m > 1:

        Kyy_mean = (
            (
                Kyy.sum()
                - np.trace(Kyy)
            )
            / (
                m * (m - 1)
            )
        )

    else:

        Kyy_mean = 0


    Kxy_mean = Kxy.mean()


    mmd_squared = (
        Kxx_mean
        + Kyy_mean
        - 2 * Kxy_mean
    )

    return float(
        max(
            mmd_squared,
            0
        )
    )


def analyze_domain_shift(
    X_current,
    X_2050,
    X_2080,
    feature_names
):

    print(
        "\n"
        + "=" * 75
    )

    print(
        "DOMAIN SHIFT ANALYSIS"
    )

    print(
        "=" * 75
    )


    n = min(
        cfg.DOMAIN_SHIFT_SAMPLE,

        len(X_current),

        len(X_2050),

        len(X_2080)
    )


    rng = np.random.default_rng(
        cfg.RANDOM_STATE
    )


    idx_current = rng.choice(
        len(X_current),
        size=n,
        replace=False
    )

    idx_2050 = rng.choice(
        len(X_2050),
        size=n,
        replace=False
    )

    idx_2080 = rng.choice(
        len(X_2080),
        size=n,
        replace=False
    )


    Xc = X_current[
        idx_current
    ]

    X50 = X_2050[
        idx_2050
    ]

    X80 = X_2080[
        idx_2080
    ]


    # --------------------------------------------------------
    # Common scaling
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_all = np.vstack([
        Xc,
        X50,
        X80
    ])

    X_all_s = scaler.fit_transform(
        X_all
    )

    Xc_s = X_all_s[
        :n
    ]

    X50_s = X_all_s[
        n:2*n
    ]

    X80_s = X_all_s[
        2*n:
    ]


    # --------------------------------------------------------
    # MMD
    # --------------------------------------------------------

    mmd_2050 = calculate_mmd(
        Xc_s,
        X50_s
    )

    mmd_2080 = calculate_mmd(
        Xc_s,
        X80_s
    )

    mmd_50_80 = calculate_mmd(
        X50_s,
        X80_s
    )


    mmd_df = pd.DataFrame({

        "Comparison": [
            "Current_vs_2050",
            "Current_vs_2080",
            "2050_vs_2080"
        ],

        "MMD_squared": [
            mmd_2050,
            mmd_2080,
            mmd_50_80
        ]
    })


    mmd_path = (
        cfg.OUTPUT_DIR
        / "domain_shift_MMD.csv"
    )

    mmd_df.to_csv(
        mmd_path,
        index=False
    )


    print(
        mmd_df.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # PCA
    # --------------------------------------------------------

    pca = PCA(
        n_components=2,
        random_state=cfg.RANDOM_STATE
    )

    emb = pca.fit_transform(
        X_all_s
    )


    labels = (
        ["Current"] * n
        + ["2050"] * n
        + ["2080"] * n
    )


    pca_df = pd.DataFrame({

        "PC1": emb[:, 0],

        "PC2": emb[:, 1],

        "Climate": labels
    })


    plt.figure(
        figsize=(9, 7)
    )

    sns.scatterplot(

        data=pca_df,

        x="PC1",

        y="PC2",

        hue="Climate",

        alpha=0.65,

        s=30
    )

    plt.title(
        "Present–Future Feature-Space Shift"
    )

    plt.xlabel(
        f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)"
    )

    plt.ylabel(
        f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)"
    )

    plt.tight_layout()


    pca_path = (
        cfg.OUTPUT_DIR
        / "domain_shift_PCA.png"
    )

    plt.savefig(
        pca_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


    print(
        f"PCA saved: {pca_path}"
    )


    return mmd_df


# ============================================================
# 22. DNN FREEZING SENSITIVITY
# ============================================================

def run_freezing_sensitivity(
    current_results,
    X_future_pool,
    y_future_pool,
    X_future_test,
    y_future_test,
    fraction,
    seed,
    year
):

    rows = []


    X_adapt, y_adapt = (
        sample_future_training_data(

            X_future_pool,

            y_future_pool,

            fraction,

            seed
        )
    )


    scaler_X = current_results[
        "scaler_X"
    ]

    scaler_y = current_results[
        "scaler_y"
    ]


    X_adapt_s = scaler_X.transform(
        X_adapt
    )

    X_test_s = scaler_X.transform(
        X_future_test
    )

    y_adapt_s = scaler_y.transform(
        y_adapt
    )


    source_dnn = current_results[
        "models"
    ]["DNN"]


    for strategy in cfg.DNN_FREEZE_STRATEGIES:

        _, pred = transfer_learn_dnn(

            source_dnn,

            X_adapt_s,

            y_adapt_s,

            X_test_s,

            scaler_y,

            strategy,

            seed
        )


        metrics = calculate_metrics(
            y_future_test,
            pred
        )


        for target in cfg.TARGETS:

            rows.append({

                "Year": year,

                "Fraction": fraction,

                "Seed": seed,

                "Freeze_Strategy": strategy,

                "Target": target,

                "R2": metrics[target]["R2"],

                "RMSE": metrics[target]["RMSE"],

                "MAE": metrics[target]["MAE"]
            })


    return pd.DataFrame(rows)


# ============================================================
# 23. SHAP
#
# SHAP is treated as a secondary interpretation analysis,
# not the main contribution.
# ============================================================

def run_shap_analysis(
    current_results,
    X_current,
    feature_names
):

    print(
        "\nRunning SHAP analysis..."
    )


    model = current_results[
        "models"
    ]["XGB"].estimators_[0]


    scaler_X = current_results[
        "scaler_X"
    ]


    n = min(
        cfg.SHAP_SAMPLE_SIZE,
        len(X_current)
    )


    rng = np.random.default_rng(
        cfg.RANDOM_STATE
    )


    indices = rng.choice(
        len(X_current),
        size=n,
        replace=False
    )


    X_sample = X_current[
        indices
    ]


    X_sample_s = scaler_X.transform(
        X_sample
    )


    try:

        explainer = shap.TreeExplainer(
            model
        )

        shap_values = (
            explainer.shap_values(
                X_sample_s
            )
        )


        mean_abs = (
            np.abs(
                shap_values
            )
            .mean(axis=0)
        )


        importance = pd.DataFrame({

            "Feature": feature_names,

            "MeanAbsSHAP": mean_abs
        }).sort_values(

            "MeanAbsSHAP",

            ascending=False
        )


        path = (
            cfg.OUTPUT_DIR
            / "SHAP_XGB_t1.csv"
        )


        importance.to_csv(
            path,
            index=False
        )


        print(
            importance.head(
                15
            ).to_string(
                index=False
            )
        )


        print(
            f"SHAP saved: {path}"
        )


        return importance


    except Exception as e:

        print(
            f"SHAP analysis failed: {e}"
        )

        return None


# ============================================================
# 24. RESULTS TABLE BUILDER
# ============================================================

def metrics_to_rows(
    results,
    year,
    experiment,
    fraction=None,
    seed=None
):

    rows = []


    for model in [
        "RF",
        "XGB",
        "DNN"
    ]:

        for target in cfg.TARGETS:

            metric = results[
                model
            ]["metrics"][target]


            rows.append({

                "Experiment": experiment,

                "Year": year,

                "Model": model,

                "Target": target,

                "Fraction": fraction,

                "Seed": seed,

                "R2": metric["R2"],

                "RMSE": metric["RMSE"],

                "MAE": metric["MAE"]
            })


    return rows


# ============================================================
# 25. MAIN PIPELINE
# ============================================================

def main():

    print(
        "\n"
        + "=" * 80
    )

    print(
        "BUILDINGCLIMATEML"
    )

    print(
        "Reliability-Aware Climate Adaptation Pipeline"
    )

    print(
        "=" * 80
    )


    # --------------------------------------------------------
    # LOAD DATA
    # --------------------------------------------------------

    df_current = load_dataset(
        cfg.CURRENT_FILE,
        "Current"
    )

    df_2050 = load_dataset(
        cfg.FILE_2050,
        "2050"
    )

    df_2080 = load_dataset(
        cfg.FILE_2080,
        "2080"
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    for df, name in [

        (df_current, "Current"),

        (df_2050, "2050"),

        (df_2080, "2080")
    ]:

        validate_targets(
            df,
            name
        )

        validate_numeric(
            df,
            name
        )


    # --------------------------------------------------------
    # ALIGN FUTURE DATA TO CURRENT FEATURE SCHEMA
    # --------------------------------------------------------

    df_2050 = align_future_schema(
        df_current,
        df_2050,
        "2050"
    )

    df_2080 = align_future_schema(
        df_current,
        df_2080,
        "2080"
    )


    # --------------------------------------------------------
    # FEATURES
    # --------------------------------------------------------

    feature_names = get_feature_names(
        df_current
    )


    print(
        "\nNumber of input features:",
        len(feature_names)
    )

    print(
        "Targets:",
        cfg.TARGETS
    )


    # --------------------------------------------------------
    # PREPARE DATA
    # --------------------------------------------------------

    X_current, y_current = prepare_xy(
        df_current,
        feature_names
    )

    X_2050, y_2050 = prepare_xy(
        df_2050,
        feature_names
    )

    X_2080, y_2080 = prepare_xy(
        df_2080,
        feature_names
    )


    print(
        "\nDataset sizes:"
    )

    print(
        "Current:",
        X_current.shape
    )

    print(
        "2050:",
        X_2050.shape
    )

    print(
        "2080:",
        X_2080.shape
    )


    # --------------------------------------------------------
    # SAVE DATASET METADATA
    # --------------------------------------------------------

    metadata = {

        "current_samples": len(
            X_current
        ),

        "future_2050_samples": len(
            X_2050
        ),

        "future_2080_samples": len(
            X_2080
        ),

        "n_features": len(
            feature_names
        ),

        "targets": cfg.TARGETS,

        "future_fractions": (
            cfg.FUTURE_FRACTIONS
        ),

        "n_repeats": cfg.N_REPEATS,

        "bootstrap_iterations": (
            cfg.BOOTSTRAP_ITERATIONS
        )
    }


    save_json(

        metadata,

        cfg.OUTPUT_DIR
        / "pipeline_metadata.json"
    )


    # ========================================================
    # SOURCE DOMAIN
    # ========================================================

    current_results = train_current_models(
        X_current,
        y_current
    )


    # --------------------------------------------------------
    # Current-domain metrics
    # --------------------------------------------------------

    current_metrics = {}


    for model in [
        "RF",
        "XGB",
        "DNN"
    ]:

        current_metrics[model] = (
            calculate_metrics(

                current_results[
                    "y_test"
                ],

                current_results[
                    "predictions"
                ][model]
            )
        )


    # ========================================================
    # FUTURE DATA FIXED TEST SPLITS
    # ========================================================

    (
        X_2050_pool,
        X_2050_test,
        y_2050_pool,
        y_2050_test
    ) = create_fixed_future_split(
        X_2050,
        y_2050,
        seed=20250
    )


    (
        X_2080_pool,
        X_2080_test,
        y_2080_pool,
        y_2080_test
    ) = create_fixed_future_split(
        X_2080,
        y_2080,
        seed=20800
    )


    # ========================================================
    # ZERO-SHOT
    # ========================================================

    print(
        "\n"
        + "=" * 80
    )

    print(
        "ZERO-SHOT FUTURE EVALUATION"
    )

    print(
        "=" * 80
    )


    zero_2050 = evaluate_zero_shot(

        current_results,

        X_2050_test,

        y_2050_test
    )


    zero_2080 = evaluate_zero_shot(

        current_results,

        X_2080_test,

        y_2080_test
    )


    # ========================================================
    # STORE ALL RESULTS
    # ========================================================

    all_metric_rows = []

    all_bootstrap_rows = []

    all_recovery_rows = []

    all_degradation_rows = []

    freezing_results = []


    # --------------------------------------------------------
    # Current baseline
    # --------------------------------------------------------

    for model in [
        "RF",
        "XGB",
        "DNN"
    ]:

        for target in cfg.TARGETS:

            m = current_metrics[
                model
            ][target]

            all_metric_rows.append({

                "Experiment": "Current",

                "Year": "Current",

                "Model": model,

                "Target": target,

                "Fraction": np.nan,

                "Seed": cfg.RANDOM_STATE,

                "R2": m["R2"],

                "RMSE": m["RMSE"],

                "MAE": m["MAE"]
            })


    # ========================================================
    # FUTURE YEARS
    # ========================================================

    for year, X_pool, y_pool, X_test, y_test, zero_results in [

        (
            "2050",
            X_2050_pool,
            y_2050_pool,
            X_2050_test,
            y_2050_test,
            zero_2050
        ),

        (
            "2080",
            X_2080_pool,
            y_2080_pool,
            X_2080_test,
            y_2080_test,
            zero_2080
        )
    ]:


        print(
            "\n"
            + "=" * 80
        )

        print(
            f"FUTURE ADAPTATION: {year}"
        )

        print(
            "=" * 80
        )


        # ----------------------------------------------------
        # Zero-shot rows
        # ----------------------------------------------------

        all_metric_rows.extend(

            metrics_to_rows(

                zero_results,

                year,

                "ZeroShot",

                fraction=0.0,

                seed=cfg.RANDOM_STATE
            )
        )


        # ----------------------------------------------------
        # Full future reference
        #
        # Train on the COMPLETE future training pool.
        # Evaluate on the SAME fixed test set used by all
        # adaptation experiments.
        # ----------------------------------------------------

        print(
            f"\nTraining full future models for {year}..."
        )


        full_results = train_full_future_models(

            X_pool,

            y_pool,

            X_test,

            y_test,

            cfg.RANDOM_STATE
        )


        all_metric_rows.extend(

            metrics_to_rows(

                full_results,

                year,

                "FullFuture",

                fraction=1.0,

                seed=cfg.RANDOM_STATE
            )
        )


        # ----------------------------------------------------
        # Data fractions
        # ----------------------------------------------------

        for fraction in cfg.FUTURE_FRACTIONS:

            if fraction == 0:

                continue

            print(
                f"\n{year} | "
                f"Future-data fraction = "
                f"{fraction:.0%}"
            )


            repeat_results = []


            for repeat_idx, seed in enumerate(
                cfg.REPEAT_SEEDS
            ):

                print(
                    f"  Repeat "
                    f"{repeat_idx + 1}/"
                    f"{len(cfg.REPEAT_SEEDS)} "
                    f"(seed={seed})"
                )


                set_global_seed(
                    seed
                )


                results = run_future_adaptation(

                    current_results,

                    X_pool,

                    y_pool,

                    X_test,

                    y_test,

                    fraction,

                    seed,

                    freeze_strategy=1
                )


                repeat_results.append(
                    results
                )


                # --------------------------------------------
                # Metric rows
                # --------------------------------------------

                all_metric_rows.extend(

                    metrics_to_rows(

                        results,

                        year,

                        "Adaptation",

                        fraction=fraction,

                        seed=seed
                    )
                )


                # --------------------------------------------
                # DNN freezing sensitivity
                #
                # Only run at 30% to avoid huge computational
                # cost.
                # --------------------------------------------

                if fraction == 0.30:

                    freezing_df = (
                        run_freezing_sensitivity(

                            current_results,

                            X_pool,

                            y_pool,

                            X_test,

                            y_test,

                            fraction,

                            seed,

                            year
                        )
                    )

                    freezing_results.append(
                        freezing_df
                    )


                # --------------------------------------------
                # Bootstrap results
                # --------------------------------------------

                for model in [
                    "RF",
                    "XGB",
                    "DNN"
                ]:

                    for target in cfg.TARGETS:

                        for metric in [
                            "R2",
                            "RMSE",
                            "MAE"
                        ]:

                            ci = results[
                                model
                            ][
                                "bootstrap"
                            ][
                                target
                            ][metric]


                            all_bootstrap_rows.append({

                                "Year": year,

                                "Fraction": fraction,

                                "Seed": seed,

                                "Model": model,

                                "Target": target,

                                "Metric": metric,

                                "Estimate": ci[
                                    "estimate"
                                ],

                                "CI_Lower": ci[
                                    "lower"
                                ],

                                "CI_Upper": ci[
                                    "upper"
                                ]
                            })


                # --------------------------------------------
                # Recovery relative to zero-shot and full
                # future reference
                # --------------------------------------------

                recovery_df = calculate_recovery(

                    zero_results["RF"]
                    if False else
                    {
                        "RF": zero_results["RF"]["metrics"],
                        "XGB": zero_results["XGB"]["metrics"],
                        "DNN": zero_results["DNN"]["metrics"]
                    },

                    {
                        model: results[
                            model
                        ]["metrics"]
                        for model in [
                            "RF",
                            "XGB",
                            "DNN"
                        ]
                    },

                    {
                        model: full_results[
                            model
                        ]["metrics"]
                        for model in [
                            "RF",
                            "XGB",
                            "DNN"
                        ]
                    }
                )


                recovery_df[
                    "Year"
                ] = year

                recovery_df[
                    "Fraction"
                ] = fraction

                recovery_df[
                    "Seed"
                ] = seed


                all_recovery_rows.append(
                    recovery_df
                )


        # ----------------------------------------------------
        # Reliability degradation
        # ----------------------------------------------------

        future_metrics_for_degradation = {

            model: zero_results[
                model
            ]["metrics"]

            for model in [
                "RF",
                "XGB",
                "DNN"
            ]
        }


        degradation_df = (
            calculate_degradation(

                current_metrics,

                future_metrics_for_degradation
            )
        )


        degradation_df[
            "Year"
        ] = year


        all_degradation_rows.append(
            degradation_df
        )


    # ========================================================
    # SAVE METRICS
    # ========================================================

    metrics_df = pd.DataFrame(
        all_metric_rows
    )


    metrics_path = (
        cfg.OUTPUT_DIR
        / "all_model_metrics.csv"
    )


    metrics_df.to_csv(
        metrics_path,
        index=False
    )


    # ========================================================
    # SAVE BOOTSTRAP
    # ========================================================

    bootstrap_df = pd.DataFrame(
        all_bootstrap_rows
    )


    bootstrap_path = (
        cfg.OUTPUT_DIR
        / "bootstrap_confidence_intervals.csv"
    )


    bootstrap_df.to_csv(
        bootstrap_path,
        index=False
    )


    # ========================================================
    # SAVE RECOVERY
    # ========================================================

    recovery_df_all = pd.concat(
        all_recovery_rows,
        ignore_index=True
    )


    recovery_path = (
        cfg.OUTPUT_DIR
        / "performance_recovery.csv"
    )


    recovery_df_all.to_csv(
        recovery_path,
        index=False
    )


    # ========================================================
    # SAVE DEGRADATION
    # ========================================================

    degradation_df_all = pd.concat(
        all_degradation_rows,
        ignore_index=True
    )


    degradation_path = (
        cfg.OUTPUT_DIR
        / "zero_shot_degradation.csv"
    )


    degradation_df_all.to_csv(
        degradation_path,
        index=False
    )


    # ========================================================
    # FREEZING SENSITIVITY
    # ========================================================

    if freezing_results:

        freezing_all = pd.concat(
            freezing_results,
            ignore_index=True
        )

        freezing_path = (
            cfg.OUTPUT_DIR
            / "DNN_freezing_sensitivity.csv"
        )

        freezing_all.to_csv(
            freezing_path,
            index=False
        )


    # ========================================================
    # DATA-EFFICIENCY SUMMARY
    # ========================================================

    adaptation_df = metrics_df[
        metrics_df[
            "Experiment"
        ] == "Adaptation"
    ].copy()


    if len(adaptation_df) > 0:

        data_efficiency = (

            adaptation_df

            .groupby(
                [
                    "Year",
                    "Fraction",
                    "Model",
                    "Target"
                ]
            )

            .agg(

                R2_mean=(
                    "R2",
                    "mean"
                ),

                R2_std=(
                    "R2",
                    "std"
                ),

                RMSE_mean=(
                    "RMSE",
                    "mean"
                ),

                RMSE_std=(
                    "RMSE",
                    "std"
                ),

                MAE_mean=(
                    "MAE",
                    "mean"
                ),

                MAE_std=(
                    "MAE",
                    "std"
                )
            )

            .reset_index()
        )


        data_efficiency_path = (
            cfg.OUTPUT_DIR
            / "data_efficiency_summary.csv"
        )


        data_efficiency.to_csv(
            data_efficiency_path,
            index=False
        )


    # ========================================================
    # DOMAIN SHIFT
    # ========================================================

    domain_shift_df = analyze_domain_shift(

        X_current,

        X_2050,

        X_2080,

        feature_names
    )


    # ========================================================
    # SHAP
    # ========================================================

    shap_results = run_shap_analysis(

        current_results,

        X_current,

        feature_names
    )


    # ========================================================
    # PRINT KEY RESULTS
    # ========================================================

    print(
        "\n"
        + "=" * 90
    )

    print(
        "ZERO-SHOT R2 RESULTS"
    )

    print(
        "=" * 90
    )


    for year, results in [

        ("2050", zero_2050),

        ("2080", zero_2080)
    ]:

        print(
            f"\n{year}"
        )

        for model in [
            "RF",
            "XGB",
            "DNN"
        ]:

            print(
                f"\n{model}"
            )

            for target in cfg.TARGETS:

                r2 = results[
                    model
                ]["metrics"][
                    target
                ]["R2"]

                print(
                    f"  {target}: "
                    f"R² = {r2:.4f}"
                )


    # ========================================================
    # DATA-EFFICIENCY PRINT
    # ========================================================

    print(
        "\n"
        + "=" * 90
    )

    print(
        "DATA-EFFICIENCY SUMMARY"
    )

    print(
        "=" * 90
    )


    if len(adaptation_df) > 0:

        summary = (

            adaptation_df

            .groupby(
                [
                    "Year",
                    "Fraction",
                    "Model"
                ]
            )["R2"]

            .mean()

            .reset_index()
        )


        print(
            summary.to_string(
                index=False
            )
        )


    # ========================================================
    # FINISHED
    # ========================================================

    print(
        "\n"
        + "=" * 90
    )

    print(
        "PIPELINE COMPLETED SUCCESSFULLY"
    )

    print(
        "=" * 90
    )

    print(
        f"\nResults saved to:\n"
        f"{cfg.OUTPUT_DIR}"
    )

    print(
        "\nKey output files:"
    )

    print(
        "1. all_model_metrics.csv"
    )

    print(
        "2. data_efficiency_summary.csv"
    )

    print(
        "3. performance_recovery.csv"
    )

    print(
        "4. zero_shot_degradation.csv"
    )

    print(
        "5. bootstrap_confidence_intervals.csv"
    )

    print(
        "6. DNN_freezing_sensitivity.csv"
    )

    print(
        "7. domain_shift_MMD.csv"
    )

    print(
        "8. domain_shift_PCA.png"
    )

    print(
        "9. SHAP_XGB_t1.csv"
    )

    print(
        "\n"
        + "=" * 90
    )


# ============================================================
# 26. RUN
# ============================================================

if __name__ == "__main__":

    main()


BUILDINGCLIMATEML
Reliability-Aware Climate Adaptation Pipeline
Current: 1826 rows × 112 columns
2050: 909 rows × 112 columns
2080: 991 rows × 112 columns

Number of input features: 107
Targets: ['t1', 't2', 't3', 't4', 't5']

Dataset sizes:
Current: (1826, 107)
2050: (909, 107)
2080: (991, 107)

SOURCE-DOMAIN TRAINING: PRESENT CLIMATE
Training Random Forest...
Training XGBoost...
Training DNN...

ZERO-SHOT FUTURE EVALUATION

FUTURE ADAPTATION: 2050

Training full future models for 2050...

2050 | Future-data fraction = 5%
  Repeat 1/5 (seed=42)
  Repeat 2/5 (seed=52)
  Repeat 3/5 (seed=62)
  Repeat 4/5 (seed=72)
  Repeat 5/5 (seed=82)

2050 | Future-data fraction = 10%
  Repeat 1/5 (seed=42)
  Repeat 2/5 (seed=52)
  Repeat 3/5 (seed=62)
  Repeat 4/5 (seed=72)
  Repeat 5/5 (seed=82)

2050 | Future-data fraction = 20%
  Repeat 1/5 (seed=42)
  Repeat 2/5 (seed=52)
  Repeat 3/5 (seed=62)
  Repeat 4/5 (seed=72)
  Repeat 5/5 (seed=82)

2050 | Future-data fraction = 30%
  Repeat 1/5 (seed=4

In [3]:
# ============================================================
# CLIMATE DOMAIN SHIFT & DATA-EFFICIENT MODEL ADAPTATION
# Applied Energy - Publication-Oriented Pipeline
#
# Experiments:
#   1. Current-domain source model
#   2. Zero-shot future prediction
#   3. Future-only training from scratch
#   4. Current + Future pooled training
#   5. DNN transfer learning
#   6. Full-future reference
#
# Models:
#   - Random Forest
#   - XGBoost
#   - Deep Neural Network
#
# Metrics:
#   - R2
#   - RMSE
#   - MAE
#   - Bias / MBE
#   - Spearman rank correlation
#   - CV(RMSE)
#
# Additional analysis:
#   - Absolute sample counts
#   - Future simulation cost
#   - Performance recovery
#   - Zero-shot degradation
#   - Transfer-learning gain
#   - Transfer vs scratch
#   - Bootstrap 95% CI
#   - MMD domain shift
#   - PCA visualization
#   - DNN freezing sensitivity
#
# IMPORTANT:
#   Current / 2050 / 2080 datasets must contain exactly:
#   P1 ... P25_WaterToAirHeatPump + t1 ... t5
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA

from scipy.stats import spearmanr

from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow.keras import (
    Model,
    Sequential
)
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    BatchNormalization,
    Input
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

GLOBAL_SEED = 42

os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)

# Optional deterministic operations
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# ============================================================
# 2. USER CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# DATA PATHS
# ------------------------------------------------------------
CURRENT_PATH = r"D:\UPM\Machine Learning\Scenario 1 Present Climate\Synthetic_data_sdv_dummies_1826_current climate.csv"

# Change these two paths to your actual files
FUTURE_2050_PATH = r"D:\UPM\Machine Learning\Scenario 2 2050\Synthetic_data_future_2050_processed.csv"
FUTURE_2080_PATH = r"D:\UPM\Machine Learning\Scenario 2 2080\Synthetic_data_future_2080_processed.csv"


# ------------------------------------------------------------
# OUTPUT DIRECTORY
# ------------------------------------------------------------
OUTPUT_DIR = Path("AppliedEnergy_ClimateDomainShift_Results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# TARGETS
# ------------------------------------------------------------
TARGETS = [
    "t1",
    "t2",
    "t3",
    "t4",
    "t5"
]


# ------------------------------------------------------------
# FUTURE FRACTIONS
#
# These fractions refer to the AVAILABLE FUTURE TRAINING POOL,
# NOT to the complete future dataset.
#
# The fixed future test set is removed first.
# ------------------------------------------------------------
FUTURE_FRACTIONS = [
    0.00,
    0.05,
    0.10,
    0.20,
    0.30,
    0.50,
    0.75,
    1.00
]


# ------------------------------------------------------------
# REPEATED EXPERIMENTS
# ------------------------------------------------------------
REPEAT_SEEDS = [
    42,
    52,
    62,
    72,
    82
]


# ------------------------------------------------------------
# DATA SPLITS
# ------------------------------------------------------------

CURRENT_TEST_SIZE = 0.20
FUTURE_TEST_SIZE = 0.20

# Validation fraction inside each training regime
VALIDATION_SIZE = 0.20


# ------------------------------------------------------------
# DNN SETTINGS
# ------------------------------------------------------------

DNN_EPOCHS = 300
DNN_BATCH_SIZE = 32

DNN_SOURCE_LR = 1e-3
DNN_ADAPT_LR = 1e-4
DNN_SCRATCH_LR = 1e-3

EARLY_STOPPING_PATIENCE = 30
LR_PATIENCE = 12


# ------------------------------------------------------------
# TREE MODELS
# ------------------------------------------------------------

RF_PARAMS = {
    "n_estimators": 350,
    "max_depth": 18,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "random_state": GLOBAL_SEED,
    "n_jobs": -1
}


XGB_PARAMS = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "objective": "reg:squarederror",
    "random_state": GLOBAL_SEED,
    "n_jobs": -1
}


# ------------------------------------------------------------
# BOOTSTRAP
# ------------------------------------------------------------

N_BOOTSTRAP = 1000


# ------------------------------------------------------------
# MMD
# ------------------------------------------------------------

MMD_SAMPLE_SIZE = 500


# ============================================================
# 3. DATA LOADING
# ============================================================

def load_dataset(path, name):
    print(f"\nLoading {name}:")
    print(path)

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\nFile not found:\n{path}\n"
            f"Please update the path in Section 2."
        )

    df = pd.read_csv(path)

    print(f"{name} shape: {df.shape}")

    return df


current_df = load_dataset(
    CURRENT_PATH,
    "Current"
)

future_2050_df = load_dataset(
    FUTURE_2050_PATH,
    "2050"
)

future_2080_df = load_dataset(
    FUTURE_2080_PATH,
    "2080"
)


# ============================================================
# 4. SCHEMA VALIDATION
# ============================================================

def validate_schema(reference_df, other_df, reference_name, other_name):

    ref_columns = list(reference_df.columns)
    other_columns = list(other_df.columns)

    if ref_columns != other_columns:

        missing = [
            c for c in ref_columns
            if c not in other_columns
        ]

        extra = [
            c for c in other_columns
            if c not in ref_columns
        ]

        raise ValueError(
            f"\nSchema mismatch between {reference_name} and {other_name}\n"
            f"Missing columns: {missing}\n"
            f"Extra columns: {extra}\n"
        )

    print(
        f"Schema verified: {reference_name} == {other_name}"
    )


validate_schema(
    current_df,
    future_2050_df,
    "Current",
    "2050"
)

validate_schema(
    current_df,
    future_2080_df,
    "Current",
    "2080"
)


# ============================================================
# 5. FEATURE IDENTIFICATION
# ============================================================

FEATURES = [
    c for c in current_df.columns
    if c not in TARGETS
]


print("\nNumber of input features:", len(FEATURES))
print("Number of targets:", len(TARGETS))

print("\nFirst features:")
print(FEATURES[:10])

print("\nLast features:")
print(FEATURES[-10:])


# ============================================================
# 6. NUMERIC VALIDATION
# ============================================================

def validate_numeric_data(df, name):

    numeric_cols = FEATURES + TARGETS

    non_numeric = []

    for col in numeric_cols:
        if not pd.api.types.is_numeric_dtype(df[col]):
            non_numeric.append(col)

    if non_numeric:
        raise TypeError(
            f"{name} contains non-numeric columns:\n"
            f"{non_numeric}"
        )

    if df[numeric_cols].isnull().any().any():

        null_counts = (
            df[numeric_cols]
            .isnull()
            .sum()
        )

        print(
            f"\nWARNING: missing values in {name}:"
        )

        print(
            null_counts[
                null_counts > 0
            ]
        )

        raise ValueError(
            f"Missing values detected in {name}. "
            f"Do not silently impute simulation outputs."
        )


validate_numeric_data(
    current_df,
    "Current"
)

validate_numeric_data(
    future_2050_df,
    "2050"
)

validate_numeric_data(
    future_2080_df,
    "2080"
)


# ============================================================
# 7. DUPLICATE CHECK
# ============================================================

def duplicate_report(df, name):

    n_duplicate_rows = df.duplicated().sum()

    n_duplicate_inputs = (
        df[FEATURES]
        .duplicated()
        .sum()
    )

    print(
        f"\n{name} duplicate check:"
    )

    print(
        f"Duplicate complete rows: {n_duplicate_rows}"
    )

    print(
        f"Duplicate feature configurations: "
        f"{n_duplicate_inputs}"
    )

    return {
        "dataset": name,
        "duplicate_rows": int(n_duplicate_rows),
        "duplicate_feature_configurations": int(
            n_duplicate_inputs
        )
    }


duplicate_reports = []

duplicate_reports.append(
    duplicate_report(
        current_df,
        "Current"
    )
)

duplicate_reports.append(
    duplicate_report(
        future_2050_df,
        "2050"
    )
)

duplicate_reports.append(
    duplicate_report(
        future_2080_df,
        "2080"
    )
)


pd.DataFrame(
    duplicate_reports
).to_csv(
    OUTPUT_DIR / "duplicate_report.csv",
    index=False
)


# ============================================================
# 8. BASIC DATASET SUMMARY
# ============================================================

dataset_summary = pd.DataFrame([
    {
        "dataset": "Current",
        "samples": len(current_df),
        "features": len(FEATURES),
        "targets": len(TARGETS)
    },
    {
        "dataset": "2050",
        "samples": len(future_2050_df),
        "features": len(FEATURES),
        "targets": len(TARGETS)
    },
    {
        "dataset": "2080",
        "samples": len(future_2080_df),
        "features": len(FEATURES),
        "targets": len(TARGETS)
    }
])

dataset_summary.to_csv(
    OUTPUT_DIR / "dataset_summary.csv",
    index=False
)

print("\nDataset summary:")
print(dataset_summary)


# ============================================================
# 9. MODEL METRICS
# ============================================================

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    results = []

    for j, target in enumerate(TARGETS):

        yt = y_true[:, j]
        yp = y_pred[:, j]

        r2 = r2_score(
            yt,
            yp
        )

        rmse = np.sqrt(
            mean_squared_error(
                yt,
                yp
            )
        )

        mae = mean_absolute_error(
            yt,
            yp
        )

        # Bias:
        # positive = overprediction
        # negative = underprediction
        bias = np.mean(
            yp - yt
        )

        # Normalized RMSE
        mean_abs_true = np.mean(
            np.abs(yt)
        )

        if mean_abs_true != 0:
            cvrmse = (
                rmse /
                mean_abs_true
            ) * 100
        else:
            cvrmse = np.nan

        # Spearman rank correlation
        try:
            rho, _ = spearmanr(
                yt,
                yp
            )
        except Exception:
            rho = np.nan

        results.append({
            "target": target,
            "R2": r2,
            "RMSE": rmse,
            "MAE": mae,
            "Bias_MBE": bias,
            "CV_RMSE_percent": cvrmse,
            "Spearman_Rho": rho
        })

    return pd.DataFrame(results)


# ============================================================
# 10. DNN ARCHITECTURE
# ============================================================

def build_dnn(
    input_dim,
    output_dim,
    learning_rate
):

    model = Sequential(
        [
            Input(
                shape=(input_dim,)
            ),

            Dense(
                256,
                activation="relu"
            ),

            BatchNormalization(),

            Dropout(
                0.30
            ),

            Dense(
                128,
                activation="relu"
            ),

            BatchNormalization(),

            Dropout(
                0.25
            ),

            Dense(
                64,
                activation="relu"
            ),

            BatchNormalization(),

            Dropout(
                0.20
            ),

            Dense(
                32,
                activation="relu"
            ),

            Dense(
                output_dim,
                activation="linear"
            )
        ]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse"
    )

    return model


# ============================================================
# 11. DNN TRAINING
# ============================================================

def train_dnn(
    X_train,
    y_train,
    X_val,
    y_val,
    learning_rate,
    epochs=DNN_EPOCHS,
    batch_size=DNN_BATCH_SIZE,
    verbose=0
):

    model = build_dnn(
        input_dim=X_train.shape[1],
        output_dim=y_train.shape[1],
        learning_rate=learning_rate
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=LR_PATIENCE,
        min_lr=1e-7,
        verbose=0
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(
            X_val,
            y_val
        ),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[
            early_stop,
            reduce_lr
        ],
        verbose=verbose
    )

    return model, history


# ============================================================
# 12. DNN TRANSFER LEARNING
# ============================================================

def clone_source_dnn(
    source_model,
    learning_rate
):

    new_model = build_dnn(
        input_dim=source_model.input_shape[1],
        output_dim=source_model.output_shape[1],
        learning_rate=learning_rate
    )

    new_model.set_weights(
        source_model.get_weights()
    )

    return new_model


def fine_tune_dnn(
    source_model,
    X_train,
    y_train,
    X_val,
    y_val,
    learning_rate=DNN_ADAPT_LR,
    freeze_layers=0,
    verbose=0
):

    model = clone_source_dnn(
        source_model,
        learning_rate
    )

    if freeze_layers > 0:

        trainable_layers = [
            layer
            for layer in model.layers
            if len(layer.weights) > 0
        ]

        for layer in trainable_layers[:freeze_layers]:
            layer.trainable = False

        model.compile(
            optimizer=Adam(
                learning_rate=learning_rate
            ),
            loss="mse"
        )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=LR_PATIENCE,
        min_lr=1e-7
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(
            X_val,
            y_val
        ),
        epochs=DNN_EPOCHS,
        batch_size=DNN_BATCH_SIZE,
        callbacks=[
            early_stop,
            reduce_lr
        ],
        verbose=verbose
    )

    return model, history


# ============================================================
# 13. SOURCE DATA SPLIT
# ============================================================

X_current = current_df[FEATURES].values.astype(
    np.float32
)

y_current = current_df[TARGETS].values.astype(
    np.float32
)


X_current_train, X_current_test, \
y_current_train, y_current_test = train_test_split(
    X_current,
    y_current,
    test_size=CURRENT_TEST_SIZE,
    random_state=GLOBAL_SEED
)


# ============================================================
# 14. SOURCE DNN SCALERS
# ============================================================

source_X_scaler = StandardScaler()

source_y_scaler = StandardScaler()


X_current_train_scaled = (
    source_X_scaler.fit_transform(
        X_current_train
    )
)

X_current_test_scaled = (
    source_X_scaler.transform(
        X_current_test
    )
)


y_current_train_scaled = (
    source_y_scaler.fit_transform(
        y_current_train
    )
)

y_current_test_scaled = (
    source_y_scaler.transform(
        y_current_test
    )
)


# ============================================================
# 15. TRAIN SOURCE DNN
# ============================================================

print(
    "\nTraining source-domain DNN..."
)

X_src_train, X_src_val, \
y_src_train, y_src_val = train_test_split(
    X_current_train_scaled,
    y_current_train_scaled,
    test_size=VALIDATION_SIZE,
    random_state=GLOBAL_SEED
)


source_dnn, source_history = train_dnn(
    X_src_train,
    y_src_train,
    X_src_val,
    y_src_val,
    learning_rate=DNN_SOURCE_LR,
    verbose=0
)


# ============================================================
# 16. SOURCE MODEL PERFORMANCE
# ============================================================

source_dnn_pred_scaled = source_dnn.predict(
    X_current_test_scaled,
    verbose=0
)

source_dnn_pred = (
    source_y_scaler
    .inverse_transform(
        source_dnn_pred_scaled
    )
)


source_dnn_metrics = calculate_metrics(
    y_current_test,
    source_dnn_pred
)

source_dnn_metrics[
    "dataset"
] = "Current"

source_dnn_metrics[
    "training_regime"
] = "Current_source"

source_dnn_metrics[
    "model"
] = "DNN"


# ============================================================
# 17. TREE SOURCE MODELS
# ============================================================

print(
    "\nTraining source-domain RF..."
)

rf_source = RandomForestRegressor(
    **RF_PARAMS
)

rf_source.fit(
    X_current_train,
    y_current_train
)


print(
    "Training source-domain XGB..."
)

xgb_source = XGBRegressor(
    **XGB_PARAMS
)

xgb_source.fit(
    X_current_train,
    y_current_train
)


rf_source_pred = rf_source.predict(
    X_current_test
)

xgb_source_pred = xgb_source.predict(
    X_current_test
)


rf_source_metrics = calculate_metrics(
    y_current_test,
    rf_source_pred
)

xgb_source_metrics = calculate_metrics(
    y_current_test,
    xgb_source_pred
)


for df, model_name in [
    (rf_source_metrics, "RF"),
    (xgb_source_metrics, "XGB")
]:

    df["dataset"] = "Current"
    df["training_regime"] = "Current_source"
    df["model"] = model_name


# ============================================================
# 18. STORE SOURCE RESULTS
# ============================================================

source_results = pd.concat(
    [
        rf_source_metrics,
        xgb_source_metrics,
        source_dnn_metrics
    ],
    ignore_index=True
)

source_results.to_csv(
    OUTPUT_DIR / "current_source_performance.csv",
    index=False
)


# ============================================================
# 19. PREPARE FUTURE DATA
# ============================================================

future_datasets = {
    "2050": future_2050_df,
    "2080": future_2080_df
}


# ============================================================
# 20. FIXED FUTURE TEST SPLITS
# ============================================================

future_splits = {}

for year, df in future_datasets.items():

    X_future = df[
        FEATURES
    ].values.astype(
        np.float32
    )

    y_future = df[
        TARGETS
    ].values.astype(
        np.float32
    )

    if year == "2050":
        split_seed = 20250
    else:
        split_seed = 20800

    X_future_pool, X_future_test, \
    y_future_pool, y_future_test = train_test_split(
        X_future,
        y_future,
        test_size=FUTURE_TEST_SIZE,
        random_state=split_seed
    )

    future_splits[year] = {
        "X_pool": X_future_pool,
        "y_pool": y_future_pool,
        "X_test": X_future_test,
        "y_test": y_future_test,
        "total": len(df),
        "pool_size": len(X_future_pool),
        "test_size": len(X_future_test)
    }

    print(
        f"\n{year}:"
    )

    print(
        f"Total simulations: {len(df)}"
    )

    print(
        f"Fixed test simulations: "
        f"{len(X_future_test)}"
    )

    print(
        f"Adaptation pool: "
        f"{len(X_future_pool)}"
    )


# ============================================================
# 21. FUTURE SAMPLE SELECTION
# ============================================================

def select_future_subset(
    X_pool,
    y_pool,
    fraction,
    seed
):

    if fraction <= 0:
        return (
            np.empty(
                (0, X_pool.shape[1]),
                dtype=np.float32
            ),
            np.empty(
                (0, y_pool.shape[1]),
                dtype=np.float32
            )
        )

    if fraction >= 1:
        return (
            X_pool.copy(),
            y_pool.copy()
        )

    n = max(
        1,
        int(
            round(
                fraction *
                len(X_pool)
            )
        )
    )

    rng = np.random.default_rng(
        seed
    )

    indices = rng.choice(
        len(X_pool),
        size=n,
        replace=False
    )

    return (
        X_pool[indices],
        y_pool[indices]
    )


# ============================================================
# 22. TRAIN RF
# ============================================================

def train_rf(
    X_train,
    y_train,
    seed
):

    params = RF_PARAMS.copy()

    params["random_state"] = seed

    model = RandomForestRegressor(
        **params
    )

    model.fit(
        X_train,
        y_train
    )

    return model


# ============================================================
# 23. TRAIN XGB
# ============================================================

def train_xgb(
    X_train,
    y_train,
    seed
):

    params = XGB_PARAMS.copy()

    params["random_state"] = seed

    model = XGBRegressor(
        **params
    )

    model.fit(
        X_train,
        y_train
    )

    return model


# ============================================================
# 24. TRAIN DNN FROM SCRATCH
# ============================================================

def train_dnn_from_scratch(
    X_train,
    y_train,
    seed
):

    tf.random.set_seed(
        seed
    )

    X_train_split, X_val, \
    y_train_split, y_val = train_test_split(
        X_train,
        y_train,
        test_size=VALIDATION_SIZE,
        random_state=seed
    )

    X_scaler = StandardScaler()

    y_scaler = StandardScaler()

    X_train_scaled = (
        X_scaler.fit_transform(
            X_train_split
        )
    )

    X_val_scaled = (
        X_scaler.transform(
            X_val
        )
    )

    y_train_scaled = (
        y_scaler.fit_transform(
            y_train_split
        )
    )

    y_val_scaled = (
        y_scaler.transform(
            y_val
        )
    )

    model, history = train_dnn(
        X_train_scaled,
        y_train_scaled,
        X_val_scaled,
        y_val_scaled,
        learning_rate=DNN_SCRATCH_LR,
        verbose=0
    )

    return (
        model,
        X_scaler,
        y_scaler,
        history
    )


# ============================================================
# 25. DNN TRANSFER LEARNING
# ============================================================

def train_dnn_transfer(
    X_future,
    y_future,
    source_model,
    source_X_scaler,
    source_y_scaler,
    seed
):

    tf.random.set_seed(
        seed
    )

    X_train, X_val, \
    y_train, y_val = train_test_split(
        X_future,
        y_future,
        test_size=VALIDATION_SIZE,
        random_state=seed
    )

    # IMPORTANT:
    # Transfer learning uses the source-domain feature and
    # target transformations.
    #
    # This keeps the representation inherited from the
    # current-climate domain consistent.

    X_train_scaled = (
        source_X_scaler.transform(
            X_train
        )
    )

    X_val_scaled = (
        source_X_scaler.transform(
            X_val
        )
    )

    y_train_scaled = (
        source_y_scaler.transform(
            y_train
        )
    )

    y_val_scaled = (
        source_y_scaler.transform(
            y_val
        )
    )

    model, history = fine_tune_dnn(
        source_model,
        X_train_scaled,
        y_train_scaled,
        X_val_scaled,
        y_val_scaled,
        learning_rate=DNN_ADAPT_LR,
        freeze_layers=0,
        verbose=0
    )

    return (
        model,
        history
    )


# ============================================================
# 26. EVALUATION HELPER
# ============================================================

def evaluate_prediction(
    y_true,
    y_pred,
    year,
    model,
    regime,
    fraction,
    repeat,
    n_future_train,
    total_future_samples,
    future_test_size
):

    metrics = calculate_metrics(
        y_true,
        y_pred
    )

    metrics[
        "year"
    ] = year

    metrics[
        "model"
    ] = model

    metrics[
        "training_regime"
    ] = regime

    metrics[
        "future_fraction"
    ] = fraction

    metrics[
        "repeat"
    ] = repeat

    metrics[
        "n_future_train"
    ] = n_future_train

    metrics[
        "total_future_samples"
    ] = total_future_samples

    metrics[
        "future_test_size"
    ] = future_test_size

    return metrics


# ============================================================
# 27. MAIN EXPERIMENT
# ============================================================

all_results = []

print(
    "\n======================================================"
)

print(
    "STARTING MAIN DOMAIN-SHIFT EXPERIMENT"
)

print(
    "======================================================"
)


for year, split in future_splits.items():

    print(
        f"\n\n================ {year} ================"
    )

    X_pool = split["X_pool"]
    y_pool = split["y_pool"]

    X_test = split["X_test"]
    y_test = split["y_test"]

    total_future = split["total"]
    future_test_size = split["test_size"]

    # --------------------------------------------------------
    # ZERO-SHOT
    # --------------------------------------------------------

    print(
        f"\n{year} - Zero-shot evaluation"
    )

    rf_zero_pred = rf_source.predict(
        X_test
    )

    xgb_zero_pred = xgb_source.predict(
        X_test
    )

    X_test_scaled = (
        source_X_scaler.transform(
            X_test
        )
    )

    dnn_zero_pred_scaled = (
        source_dnn.predict(
            X_test_scaled,
            verbose=0
        )
    )

    dnn_zero_pred = (
        source_y_scaler
        .inverse_transform(
            dnn_zero_pred_scaled
        )
    )

    all_results.append(
        evaluate_prediction(
            y_test,
            rf_zero_pred,
            year,
            "RF",
            "zero_shot",
            0.0,
            0,
            0,
            total_future,
            future_test_size
        )
    )

    all_results.append(
        evaluate_prediction(
            y_test,
            xgb_zero_pred,
            year,
            "XGB",
            "zero_shot",
            0.0,
            0,
            0,
            total_future,
            future_test_size
        )
    )

    all_results.append(
        evaluate_prediction(
            y_test,
            dnn_zero_pred,
            year,
            "DNN",
            "zero_shot",
            0.0,
            0,
            0,
            total_future,
            future_test_size
        )
    )

    # --------------------------------------------------------
    # FULL FUTURE REFERENCE
    # --------------------------------------------------------

    print(
        f"\n{year} - Full-future reference"
    )

    rf_full = train_rf(
        X_pool,
        y_pool,
        GLOBAL_SEED
    )

    xgb_full = train_xgb(
        X_pool,
        y_pool,
        GLOBAL_SEED
    )

    rf_full_pred = rf_full.predict(
        X_test
    )

    xgb_full_pred = xgb_full.predict(
        X_test
    )

    all_results.append(
        evaluate_prediction(
            y_test,
            rf_full_pred,
            year,
            "RF",
            "full_future",
            1.0,
            0,
            len(X_pool),
            total_future,
            future_test_size
        )
    )

    all_results.append(
        evaluate_prediction(
            y_test,
            xgb_full_pred,
            year,
            "XGB",
            "full_future",
            1.0,
            0,
            len(X_pool),
            total_future,
            future_test_size
        )
    )

    # Full-future DNN scratch
    (
        dnn_full,
        full_X_scaler,
        full_y_scaler,
        _
    ) = train_dnn_from_scratch(
        X_pool,
        y_pool,
        GLOBAL_SEED
    )

    X_test_full_scaled = (
        full_X_scaler.transform(
            X_test
        )
    )

    dnn_full_pred_scaled = (
        dnn_full.predict(
            X_test_full_scaled,
            verbose=0
        )
    )

    dnn_full_pred = (
        full_y_scaler.inverse_transform(
            dnn_full_pred_scaled
        )
    )

    all_results.append(
        evaluate_prediction(
            y_test,
            dnn_full_pred,
            year,
            "DNN",
            "full_future",
            1.0,
            0,
            len(X_pool),
            total_future,
            future_test_size
        )
    )

    # --------------------------------------------------------
    # FRACTION EXPERIMENTS
    # --------------------------------------------------------

    for fraction in FUTURE_FRACTIONS:

        if fraction == 0:
            continue

        print(
            f"\n{year} | Future fraction = "
            f"{fraction:.0%}"
        )

        for repeat, seed in enumerate(
            REPEAT_SEEDS,
            start=1
        ):

            print(
                f"  Repeat {repeat}/"
                f"{len(REPEAT_SEEDS)}"
            )

            X_future_train, y_future_train = (
                select_future_subset(
                    X_pool,
                    y_pool,
                    fraction,
                    seed
                )
            )

            n_future_train = (
                len(X_future_train)
            )

            # =================================================
            # A. FUTURE-ONLY SCRATCH
            # =================================================

            # RF
            rf_future = train_rf(
                X_future_train,
                y_future_train,
                seed
            )

            rf_future_pred = (
                rf_future.predict(
                    X_test
                )
            )

            all_results.append(
                evaluate_prediction(
                    y_test,
                    rf_future_pred,
                    year,
                    "RF",
                    "future_only",
                    fraction,
                    repeat,
                    n_future_train,
                    total_future,
                    future_test_size
                )
            )

            # XGB
            xgb_future = train_xgb(
                X_future_train,
                y_future_train,
                seed
            )

            xgb_future_pred = (
                xgb_future.predict(
                    X_test
                )
            )

            all_results.append(
                evaluate_prediction(
                    y_test,
                    xgb_future_pred,
                    year,
                    "XGB",
                    "future_only",
                    fraction,
                    repeat,
                    n_future_train,
                    total_future,
                    future_test_size
                )
            )

            # DNN scratch
            (
                dnn_scratch,
                scratch_X_scaler,
                scratch_y_scaler,
                _
            ) = train_dnn_from_scratch(
                X_future_train,
                y_future_train,
                seed
            )

            X_test_scratch_scaled = (
                scratch_X_scaler.transform(
                    X_test
                )
            )

            dnn_scratch_pred_scaled = (
                dnn_scratch.predict(
                    X_test_scratch_scaled,
                    verbose=0
                )
            )

            dnn_scratch_pred = (
                scratch_y_scaler
                .inverse_transform(
                    dnn_scratch_pred_scaled
                )
            )

            all_results.append(
                evaluate_prediction(
                    y_test,
                    dnn_scratch_pred,
                    year,
                    "DNN",
                    "future_only",
                    fraction,
                    repeat,
                    n_future_train,
                    total_future,
                    future_test_size
                )
            )

            # =================================================
            # B. POOLED CURRENT + FUTURE
            # =================================================

            X_pooled = np.vstack(
                [
                    X_current_train,
                    X_future_train
                ]
            )

            y_pooled = np.vstack(
                [
                    y_current_train,
                    y_future_train
                ]
            )

            # RF
            rf_pooled = train_rf(
                X_pooled,
                y_pooled,
                seed
            )

            rf_pooled_pred = (
                rf_pooled.predict(
                    X_test
                )
            )

            all_results.append(
                evaluate_prediction(
                    y_test,
                    rf_pooled_pred,
                    year,
                    "RF",
                    "pooled_current_future",
                    fraction,
                    repeat,
                    n_future_train,
                    total_future,
                    future_test_size
                )
            )

            # XGB
            xgb_pooled = train_xgb(
                X_pooled,
                y_pooled,
                seed
            )

            xgb_pooled_pred = (
                xgb_pooled.predict(
                    X_test
                )
            )

            all_results.append(
                evaluate_prediction(
                    y_test,
                    xgb_pooled_pred,
                    year,
                    "XGB",
                    "pooled_current_future",
                    fraction,
                    repeat,
                    n_future_train,
                    total_future,
                    future_test_size
                )
            )

            # =================================================
            # C. DNN TRANSFER LEARNING
            # =================================================

            (
                dnn_transfer,
                transfer_history
            ) = train_dnn_transfer(
                X_future_train,
                y_future_train,
                source_dnn,
                source_X_scaler,
                source_y_scaler,
                seed
            )

            X_test_transfer_scaled = (
                source_X_scaler.transform(
                    X_test
                )
            )

            dnn_transfer_pred_scaled = (
                dnn_transfer.predict(
                    X_test_transfer_scaled,
                    verbose=0
                )
            )

            dnn_transfer_pred = (
                source_y_scaler
                .inverse_transform(
                    dnn_transfer_pred_scaled
                )
            )

            all_results.append(
                evaluate_prediction(
                    y_test,
                    dnn_transfer_pred,
                    year,
                    "DNN",
                    "transfer_learning",
                    fraction,
                    repeat,
                    n_future_train,
                    total_future,
                    future_test_size
                )
            )


# ============================================================
# 28. COMBINE RESULTS
# ============================================================

results_df = pd.concat(
    all_results,
    ignore_index=True
)


results_df.to_csv(
    OUTPUT_DIR / "all_model_metrics.csv",
    index=False
)


print(
    "\nMain results saved:"
)

print(
    OUTPUT_DIR /
    "all_model_metrics.csv"
)


# ============================================================
# 29. AGGREGATED RESULTS
# ============================================================

group_columns = [
    "year",
    "model",
    "training_regime",
    "future_fraction",
    "target"
]

metric_columns = [
    "R2",
    "RMSE",
    "MAE",
    "Bias_MBE",
    "CV_RMSE_percent",
    "Spearman_Rho"
]


aggregated_results = (
    results_df
    .groupby(group_columns)[metric_columns]
    .agg(
        [
            "mean",
            "std"
        ]
    )
    .reset_index()
)


aggregated_results.to_csv(
    OUTPUT_DIR /
    "aggregated_results_mean_std.csv",
    index=False
)


# ============================================================
# 30. ABSOLUTE SAMPLE / SIMULATION COST TABLE
# ============================================================

cost_rows = []

for year, split in future_splits.items():

    pool_size = split["pool_size"]
    total_size = split["total"]
    test_size = split["test_size"]

    for fraction in FUTURE_FRACTIONS:

        n_train = int(
            round(
                fraction *
                pool_size
            )
        )

        cost_rows.append(
            {
                "year": year,
                "total_future_simulations": total_size,
                "fixed_future_test_simulations": test_size,
                "future_training_pool": pool_size,
                "future_fraction": fraction,
                "future_training_simulations": n_train,
                "additional_future_simulations_relative_to_zero_shot": n_train
            }
        )


cost_df = pd.DataFrame(
    cost_rows
)


cost_df.to_csv(
    OUTPUT_DIR /
    "future_simulation_cost.csv",
    index=False
)


# ============================================================
# 31. PERFORMANCE RECOVERY
# ============================================================

def calculate_recovery(
    zero_value,
    adapted_value,
    full_value,
    metric="R2"
):

    if metric == "R2":

        denominator = (
            full_value -
            zero_value
        )

        if abs(denominator) < 1e-12:
            return np.nan

        return (
            adapted_value -
            zero_value
        ) / denominator * 100

    else:

        # For error metrics, lower is better.
        denominator = (
            zero_value -
            full_value
        )

        if abs(denominator) < 1e-12:
            return np.nan

        return (
            zero_value -
            adapted_value
        ) / denominator * 100


recovery_rows = []


# Mean metrics over repeats
mean_results = (
    results_df
    .groupby(
        [
            "year",
            "model",
            "training_regime",
            "future_fraction",
            "target"
        ],
        as_index=False
    )[
        [
            "R2",
            "RMSE",
            "MAE",
            "Bias_MBE"
        ]
    ]
    .mean()
)


for year in ["2050", "2080"]:

    for model in ["RF", "XGB", "DNN"]:

        for target in TARGETS:

            zero_row = mean_results[
                (mean_results["year"] == year) &
                (mean_results["model"] == model) &
                (mean_results["training_regime"] == "zero_shot") &
                (mean_results["target"] == target)
            ]

            full_row = mean_results[
                (mean_results["year"] == year) &
                (mean_results["model"] == model) &
                (mean_results["training_regime"] == "full_future") &
                (mean_results["target"] == target)
            ]

            if (
                zero_row.empty or
                full_row.empty
            ):
                continue

            zero_r2 = (
                zero_row["R2"].iloc[0]
            )

            full_r2 = (
                full_row["R2"].iloc[0]
            )

            zero_rmse = (
                zero_row["RMSE"].iloc[0]
            )

            full_rmse = (
                full_row["RMSE"].iloc[0]
            )

            zero_mae = (
                zero_row["MAE"].iloc[0]
            )

            full_mae = (
                full_row["MAE"].iloc[0]
            )

            for regime in [
                "future_only",
                "pooled_current_future",
                "transfer_learning"
            ]:

                adapted = mean_results[
                    (mean_results["year"] == year) &
                    (mean_results["model"] == model) &
                    (mean_results["training_regime"] == regime) &
                    (mean_results["target"] == target)
                ]

                if adapted.empty:
                    continue

                adapted_r2 = (
                    adapted["R2"].mean()
                )

                adapted_rmse = (
                    adapted["RMSE"].mean()
                )

                adapted_mae = (
                    adapted["MAE"].mean()
                )

                recovery_rows.append(
                    {
                        "year": year,
                        "model": model,
                        "target": target,
                        "training_regime": regime,
                        "future_fraction": adapted["future_fraction"].mean(),
                        "R2_zero_shot": zero_r2,
                        "R2_adapted": adapted_r2,
                        "R2_full_future": full_r2,
                        "R2_recovery_percent":
                            calculate_recovery(
                                zero_r2,
                                adapted_r2,
                                full_r2,
                                "R2"
                            ),
                        "RMSE_zero_shot": zero_rmse,
                        "RMSE_adapted": adapted_rmse,
                        "RMSE_full_future": full_rmse,
                        "RMSE_recovery_percent":
                            calculate_recovery(
                                zero_rmse,
                                adapted_rmse,
                                full_rmse,
                                "RMSE"
                            ),
                        "MAE_zero_shot": zero_mae,
                        "MAE_adapted": adapted_mae,
                        "MAE_full_future": full_mae,
                        "MAE_recovery_percent":
                            calculate_recovery(
                                zero_mae,
                                adapted_mae,
                                full_mae,
                                "MAE"
                            )
                    }
                )


recovery_df = pd.DataFrame(
    recovery_rows
)


recovery_df.to_csv(
    OUTPUT_DIR /
    "performance_recovery.csv",
    index=False
)


# ============================================================
# 32. ZERO-SHOT DEGRADATION
# ============================================================

degradation_rows = []

for year in ["2050", "2080"]:

    for model in ["RF", "XGB", "DNN"]:

        for target in TARGETS:

            current_row = source_results[
                (source_results["model"] == model) &
                (source_results["target"] == target)
            ]

            future_row = mean_results[
                (mean_results["year"] == year) &
                (mean_results["model"] == model) &
                (mean_results["training_regime"] == "zero_shot") &
                (mean_results["target"] == target)
            ]

            if (
                current_row.empty or
                future_row.empty
            ):
                continue

            current_r2 = (
                current_row["R2"].iloc[0]
            )

            future_r2 = (
                future_row["R2"].iloc[0]
            )

            degradation_rows.append(
                {
                    "year": year,
                    "model": model,
                    "target": target,
                    "current_R2": current_r2,
                    "future_zero_shot_R2": future_r2,
                    "R2_change": future_r2 - current_r2,
                    "relative_R2_change_percent":
                        (
                            (future_r2 - current_r2)
                            /
                            abs(current_r2)
                            * 100
                            if abs(current_r2) > 1e-12
                            else np.nan
                        )
                }
            )


degradation_df = pd.DataFrame(
    degradation_rows
)


degradation_df.to_csv(
    OUTPUT_DIR /
    "zero_shot_degradation.csv",
    index=False
)


# ============================================================
# 33. TRANSFER LEARNING VS FUTURE-ONLY
# ============================================================

transfer_comparison_rows = []


for year in ["2050", "2080"]:

    for target in TARGETS:

        tl = mean_results[
            (mean_results["year"] == year) &
            (mean_results["model"] == "DNN") &
            (mean_results["training_regime"] == "transfer_learning") &
            (mean_results["target"] == target)
        ]

        scratch = mean_results[
            (mean_results["year"] == year) &
            (mean_results["model"] == "DNN") &
            (mean_results["training_regime"] == "future_only") &
            (mean_results["target"] == target)
        ]

        if tl.empty or scratch.empty:
            continue

        for fraction in FUTURE_FRACTIONS:

            if fraction == 0:
                continue

            tl_f = tl[
                np.isclose(
                    tl["future_fraction"],
                    fraction
                )
            ]

            scratch_f = scratch[
                np.isclose(
                    scratch["future_fraction"],
                    fraction
                )
            ]

            if tl_f.empty or scratch_f.empty:
                continue

            transfer_comparison_rows.append(
                {
                    "year": year,
                    "target": target,
                    "future_fraction": fraction,
                    "DNN_transfer_R2":
                        tl_f["R2"].mean(),
                    "DNN_scratch_R2":
                        scratch_f["R2"].mean(),
                    "DNN_transfer_RMSE":
                        tl_f["RMSE"].mean(),
                    "DNN_scratch_RMSE":
                        scratch_f["RMSE"].mean(),
                    "DNN_transfer_MAE":
                        tl_f["MAE"].mean(),
                    "DNN_scratch_MAE":
                        scratch_f["MAE"].mean(),
                    "R2_transfer_minus_scratch":
                        (
                            tl_f["R2"].mean()
                            -
                            scratch_f["R2"].mean()
                        )
                }
            )


transfer_comparison_df = pd.DataFrame(
    transfer_comparison_rows
)


transfer_comparison_df.to_csv(
    OUTPUT_DIR /
    "DNN_transfer_vs_scratch.csv",
    index=False
)


# ============================================================
# 34. POOLED VS FUTURE-ONLY
# ============================================================

pooled_comparison_rows = []


for year in ["2050", "2080"]:

    for model in ["RF", "XGB"]:

        for target in TARGETS:

            pooled = mean_results[
                (mean_results["year"] == year) &
                (mean_results["model"] == model) &
                (mean_results["training_regime"] == "pooled_current_future") &
                (mean_results["target"] == target)
            ]

            future_only = mean_results[
                (mean_results["year"] == year) &
                (mean_results["model"] == model) &
                (mean_results["training_regime"] == "future_only") &
                (mean_results["target"] == target)
            ]

            for fraction in FUTURE_FRACTIONS:

                if fraction == 0:
                    continue

                pooled_f = pooled[
                    np.isclose(
                        pooled["future_fraction"],
                        fraction
                    )
                ]

                future_f = future_only[
                    np.isclose(
                        future_only["future_fraction"],
                        fraction
                    )
                ]

                if pooled_f.empty or future_f.empty:
                    continue

                pooled_comparison_rows.append(
                    {
                        "year": year,
                        "model": model,
                        "target": target,
                        "future_fraction": fraction,
                        "pooled_R2":
                            pooled_f["R2"].mean(),
                        "future_only_R2":
                            future_f["R2"].mean(),
                        "pooled_RMSE":
                            pooled_f["RMSE"].mean(),
                        "future_only_RMSE":
                            future_f["RMSE"].mean(),
                        "pooled_MAE":
                            pooled_f["MAE"].mean(),
                        "future_only_MAE":
                            future_f["MAE"].mean()
                    }
                )


pooled_comparison_df = pd.DataFrame(
    pooled_comparison_rows
)


pooled_comparison_df.to_csv(
    OUTPUT_DIR /
    "pooled_vs_future_only.csv",
    index=False
)


# ============================================================
# 35. BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

def bootstrap_metric_ci(
    y_true,
    y_pred,
    metric_name,
    n_bootstrap=1000,
    seed=42
):

    rng = np.random.default_rng(
        seed
    )

    n = len(y_true)

    values = []

    for _ in range(
        n_bootstrap
    ):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        yt = y_true[idx]
        yp = y_pred[idx]

        if metric_name == "R2":

            value = r2_score(
                yt,
                yp
            )

        elif metric_name == "RMSE":

            value = np.sqrt(
                mean_squared_error(
                    yt,
                    yp
                )
            )

        elif metric_name == "MAE":

            value = mean_absolute_error(
                yt,
                yp
            )

        elif metric_name == "Bias_MBE":

            value = np.mean(
                yp - yt
            )

        else:
            raise ValueError(
                metric_name
            )

        values.append(
            value
        )

    values = np.asarray(
        values
    )

    return {
        "mean": np.mean(values),
        "lower_95": np.percentile(
            values,
            2.5
        ),
        "upper_95": np.percentile(
            values,
            97.5
        )
    }


# Bootstrap the major final configurations
bootstrap_rows = []


# To avoid excessive computation, use the first repeat
# for the representative final comparison.

for year, split in future_splits.items():

    X_test = split["X_test"]
    y_test = split["y_test"]

    # Zero-shot
    predictions = {
        "RF_zero_shot": rf_source.predict(
            X_test
        ),
        "XGB_zero_shot": xgb_source.predict(
            X_test
        ),
        "DNN_zero_shot": (
            source_y_scaler.inverse_transform(
                source_dnn.predict(
                    source_X_scaler.transform(
                        X_test
                    ),
                    verbose=0
                )
            )
        )
    }

    for model_name, pred in predictions.items():

        model = model_name.split("_")[0]

        for j, target in enumerate(TARGETS):

            yt = y_test[:, j]
            yp = pred[:, j]

            for metric in [
                "R2",
                "RMSE",
                "MAE",
                "Bias_MBE"
            ]:

                ci = bootstrap_metric_ci(
                    yt,
                    yp,
                    metric,
                    N_BOOTSTRAP,
                    GLOBAL_SEED
                )

                bootstrap_rows.append(
                    {
                        "year": year,
                        "model": model,
                        "configuration": model_name,
                        "target": target,
                        "metric": metric,
                        "mean": ci["mean"],
                        "lower_95": ci["lower_95"],
                        "upper_95": ci["upper_95"]
                    }
                )


bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

bootstrap_df.to_csv(
    OUTPUT_DIR /
    "bootstrap_confidence_intervals.csv",
    index=False
)


# ============================================================
# 36. DOMAIN SHIFT: STANDARDIZED MMD
# ============================================================

def rbf_kernel(
    X,
    Y,
    gamma
):

    X_norm = (
        np.sum(
            X ** 2,
            axis=1
        )[:, None]
    )

    Y_norm = (
        np.sum(
            Y ** 2,
            axis=1
        )[None, :]
    )

    distances = (
        X_norm +
        Y_norm -
        2 * np.dot(
            X,
            Y.T
        )
    )

    return np.exp(
        -gamma *
        distances
    )


def median_heuristic_gamma(
    X,
    max_samples=500,
    seed=42
):

    rng = np.random.default_rng(
        seed
    )

    if len(X) > max_samples:

        idx = rng.choice(
            len(X),
            max_samples,
            replace=False
        )

        X = X[idx]

    # pairwise squared distances
    norms = np.sum(
        X ** 2,
        axis=1
    )

    distances = (
        norms[:, None]
        +
        norms[None, :]
        -
        2 * np.dot(
            X,
            X.T
        )
    )

    distances = distances[
        distances > 0
    ]

    if len(distances) == 0:
        return 1.0

    median_distance = np.median(
        distances
    )

    if median_distance <= 0:
        return 1.0

    return 1.0 / (
        2 *
        median_distance
    )


def compute_mmd(
    X,
    Y,
    seed=42
):

    rng = np.random.default_rng(
        seed
    )

    if len(X) > MMD_SAMPLE_SIZE:

        X = X[
            rng.choice(
                len(X),
                MMD_SAMPLE_SIZE,
                replace=False
            )
        ]

    if len(Y) > MMD_SAMPLE_SIZE:

        Y = Y[
            rng.choice(
                len(Y),
                MMD_SAMPLE_SIZE,
                replace=False
            )
        ]

    scaler = StandardScaler()

    combined = np.vstack(
        [X, Y]
    )

    combined_scaled = scaler.fit_transform(
        combined
    )

    Xs = combined_scaled[
        :len(X)
    ]

    Ys = combined_scaled[
        len(X):
    ]

    gamma = median_heuristic_gamma(
        combined_scaled,
        seed=seed
    )

    Kxx = rbf_kernel(
        Xs,
        Xs,
        gamma
    )

    Kyy = rbf_kernel(
        Ys,
        Ys,
        gamma
    )

    Kxy = rbf_kernel(
        Xs,
        Ys,
        gamma
    )

    # Unbiased MMD^2
    n = len(Xs)
    m = len(Ys)

    Kxx_sum = (
        np.sum(Kxx) -
        np.trace(Kxx)
    )

    Kyy_sum = (
        np.sum(Kyy) -
        np.trace(Kyy)
    )

    Kxy_sum = np.sum(
        Kxy
    )

    mmd2 = (
        Kxx_sum /
        (n * (n - 1))
        +
        Kyy_sum /
        (m * (m - 1))
        -
        2 *
        Kxy_sum /
        (n * m)
    )

    return float(
        max(
            mmd2,
            0
        )
    )


# ============================================================
# 37. MMD RESULTS
# ============================================================

mmd_rows = []


for year, df in future_datasets.items():

    X_future = df[
        FEATURES
    ].values.astype(
        np.float32
    )

    mmd_value = compute_mmd(
        X_current,
        X_future,
        seed=GLOBAL_SEED
    )

    mmd_rows.append(
        {
            "comparison": f"Current_vs_{year}",
            "year": year,
            "MMD_squared": mmd_value
        }
    )


mmd_df = pd.DataFrame(
    mmd_rows
)


mmd_df.to_csv(
    OUTPUT_DIR /
    "domain_shift_MMD.csv",
    index=False
)


# ============================================================
# 38. PCA DOMAIN VISUALIZATION
# ============================================================

rng = np.random.default_rng(
    GLOBAL_SEED
)

pca_frames = []

for name, df in [
    ("Current", current_df),
    ("2050", future_2050_df),
    ("2080", future_2080_df)
]:

    X = df[
        FEATURES
    ].values.astype(
        np.float32
    )

    if len(X) > 500:

        idx = rng.choice(
            len(X),
            500,
            replace=False
        )

        X = X[idx]

    pca_frames.append(
        (
            name,
            X
        )
    )


combined_X = np.vstack(
    [
        x
        for _, x in pca_frames
    ]
)


pca_scaler = StandardScaler()

combined_scaled = (
    pca_scaler.fit_transform(
        combined_X
    )
)


pca = PCA(
    n_components=2,
    random_state=GLOBAL_SEED
)

combined_pca = pca.fit_transform(
    combined_scaled
)


plt.figure(
    figsize=(9, 7)
)

start = 0

for name, X in pca_frames:

    end = (
        start +
        len(X)
    )

    coords = combined_pca[
        start:end
    ]

    plt.scatter(
        coords[:, 0],
        coords[:, 1],
        alpha=0.55,
        s=18,
        label=name
    )

    start = end


plt.xlabel(
    "Principal Component 1"
)

plt.ylabel(
    "Principal Component 2"
)

plt.title(
    "Input-Space Distribution Across Climate Domains"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR /
    "domain_shift_PCA.png",
    dpi=300
)

plt.close()


# ============================================================
# 39. TARGET DISTRIBUTION SHIFT
#
# This is important because input-space MMD alone may not
# capture response/conditional shift.
# ============================================================

target_shift_rows = []


for year, df in future_datasets.items():

    for target in TARGETS:

        current_values = (
            current_df[target]
            .values
        )

        future_values = (
            df[target]
            .values
        )

        target_shift_rows.append(
            {
                "year": year,
                "target": target,
                "current_mean":
                    np.mean(current_values),
                "future_mean":
                    np.mean(future_values),
                "current_std":
                    np.std(current_values),
                "future_std":
                    np.std(future_values),
                "mean_difference":
                    (
                        np.mean(future_values)
                        -
                        np.mean(current_values)
                    ),
                "relative_mean_change_percent":
                    (
                        (
                            np.mean(future_values)
                            -
                            np.mean(current_values)
                        )
                        /
                        abs(
                            np.mean(current_values)
                        )
                        * 100
                        if abs(
                            np.mean(current_values)
                        ) > 1e-12
                        else np.nan
                    )
            }
        )


target_shift_df = pd.DataFrame(
    target_shift_rows
)


target_shift_df.to_csv(
    OUTPUT_DIR /
    "target_distribution_shift.csv",
    index=False
)


# ============================================================
# 40. DNN FREEZING SENSITIVITY
# ============================================================

FREEZE_LEVELS = [
    0,
    2,
    4
]

FREEZING_FRACTION = 0.30

freezing_rows = []


for year, split in future_splits.items():

    X_pool = split["X_pool"]
    y_pool = split["y_pool"]

    X_test = split["X_test"]
    y_test = split["y_test"]

    X_future_train, y_future_train = (
        select_future_subset(
            X_pool,
            y_pool,
            FREEZING_FRACTION,
            GLOBAL_SEED
        )
    )

    X_train, X_val, \
    y_train, y_val = train_test_split(
        X_future_train,
        y_future_train,
        test_size=VALIDATION_SIZE,
        random_state=GLOBAL_SEED
    )

    X_train_scaled = (
        source_X_scaler.transform(
            X_train
        )
    )

    X_val_scaled = (
        source_X_scaler.transform(
            X_val
        )
    )

    y_train_scaled = (
        source_y_scaler.transform(
            y_train
        )
    )

    y_val_scaled = (
        source_y_scaler.transform(
            y_val
        )
    )

    X_test_scaled = (
        source_X_scaler.transform(
            X_test
        )
    )

    for freeze_level in FREEZE_LEVELS:

        model = clone_source_dnn(
            source_dnn,
            DNN_ADAPT_LR
        )

        if freeze_level > 0:

            trainable_layers = [
                layer
                for layer in model.layers
                if len(layer.weights) > 0
            ]

            for layer in trainable_layers[
                :freeze_level
            ]:

                layer.trainable = False

        model.compile(
            optimizer=Adam(
                learning_rate=DNN_ADAPT_LR
            ),
            loss="mse"
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True
        )

        model.fit(
            X_train_scaled,
            y_train_scaled,
            validation_data=(
                X_val_scaled,
                y_val_scaled
            ),
            epochs=DNN_EPOCHS,
            batch_size=DNN_BATCH_SIZE,
            callbacks=[
                early_stop
            ],
            verbose=0
        )

        pred_scaled = (
            model.predict(
                X_test_scaled,
                verbose=0
            )
        )

        pred = (
            source_y_scaler
            .inverse_transform(
                pred_scaled
            )
        )

        metrics = calculate_metrics(
            y_test,
            pred
        )

        metrics[
            "year"
        ] = year

        metrics[
            "freeze_level"
        ] = freeze_level

        metrics[
            "future_fraction"
        ] = FREEZING_FRACTION

        freezing_rows.append(
            metrics
        )


freezing_df = pd.concat(
    freezing_rows,
    ignore_index=True
)


freezing_df.to_csv(
    OUTPUT_DIR /
    "DNN_freezing_sensitivity.csv",
    index=False
)


# ============================================================
# 41. MODEL RANKING CONSISTENCY
# ============================================================

ranking_rows = []


for year in ["2050", "2080"]:

    for target in TARGETS:

        subset = mean_results[
            (mean_results["year"] == year) &
            (mean_results["target"] == target)
        ]

        if subset.empty:
            continue

        # Average R2 by model/regime/fraction
        ranking = (
            subset
            .sort_values(
                "R2",
                ascending=False
            )
        )

        for rank, (_, row) in enumerate(
            ranking.iterrows(),
            start=1
        ):

            ranking_rows.append(
                {
                    "year": year,
                    "target": target,
                    "model": row["model"],
                    "training_regime":
                        row["training_regime"],
                    "future_fraction":
                        row["future_fraction"],
                    "R2_rank": rank
                }
            )


ranking_df = pd.DataFrame(
    ranking_rows
)


ranking_df.to_csv(
    OUTPUT_DIR /
    "model_ranking.csv",
    index=False
)


# ============================================================
# 42. PERFORMANCE VS SIMULATION COST PLOTS
# ============================================================

for year in ["2050", "2080"]:

    for target in TARGETS:

        subset = mean_results[
            (mean_results["year"] == year) &
            (mean_results["target"] == target)
        ]

        if subset.empty:
            continue

        plt.figure(
            figsize=(9, 6)
        )

        for model, regime in [
            ("RF", "future_only"),
            ("RF", "pooled_current_future"),
            ("XGB", "future_only"),
            ("XGB", "pooled_current_future"),
            ("DNN", "future_only"),
            ("DNN", "transfer_learning")
        ]:

            data = subset[
                (subset["model"] == model) &
                (subset["training_regime"] == regime)
            ]

            if data.empty:
                continue

            cost_lookup = cost_df[
                cost_df["year"] == year
            ][
                [
                    "future_fraction",
                    "future_training_simulations"
                ]
            ]

            merged = data.merge(
                cost_lookup,
                on="future_fraction",
                how="left"
            )

            plt.plot(
                merged[
                    "future_training_simulations"
                ],
                merged["R2"],
                marker="o",
                label=f"{model} - {regime}"
            )

        plt.xlabel(
            "Additional Future Simulations Used for Training"
        )

        plt.ylabel(
            "R² on Independent Future Test Set"
        )

        plt.title(
            f"{year} – Predictive Recovery vs Future Simulation Cost – {target}"
        )

        plt.legend(
            fontsize=8
        )

        plt.tight_layout()

        filename = (
            f"R2_vs_simulation_cost_"
            f"{year}_{target}.png"
        )

        plt.savefig(
            OUTPUT_DIR / filename,
            dpi=300
        )

        plt.close()


# ============================================================
# 43. PERFORMANCE CURVES
# ============================================================

for year in ["2050", "2080"]:

    for target in TARGETS:

        subset = mean_results[
            (mean_results["year"] == year) &
            (mean_results["target"] == target)
        ]

        plt.figure(
            figsize=(9, 6)
        )

        for model, regime in [
            ("RF", "future_only"),
            ("RF", "pooled_current_future"),
            ("XGB", "future_only"),
            ("XGB", "pooled_current_future"),
            ("DNN", "future_only"),
            ("DNN", "transfer_learning")
        ]:

            data = subset[
                (subset["model"] == model) &
                (subset["training_regime"] == regime)
            ]

            if data.empty:
                continue

            plt.plot(
                data["future_fraction"] * 100,
                data["R2"],
                marker="o",
                label=f"{model} - {regime}"
            )

        plt.axhline(
            0,
            linestyle="--",
            linewidth=1
        )

        plt.xlabel(
            "Future Training Data (%)"
        )

        plt.ylabel(
            "R²"
        )

        plt.title(
            f"{year} – Data-Efficiency Curve – {target}"
        )

        plt.legend(
            fontsize=8
        )

        plt.tight_layout()

        plt.savefig(
            OUTPUT_DIR /
            f"data_efficiency_R2_{year}_{target}.png",
            dpi=300
        )

        plt.close()


# ============================================================
# 44. SHAP / XGB INTERPRETABILITY
# ============================================================

# SHAP is intentionally treated as a secondary analysis.
# The primary contribution is domain adaptation/data efficiency.

try:

    import shap

    # Train representative full-future XGB
    # separately for 2050 and 2080.

    for year, split in future_splits.items():

        X_pool = split["X_pool"]
        y_pool = split["y_pool"]

        model = train_xgb(
            X_pool,
            y_pool,
            GLOBAL_SEED
        )

        # Analyze t1 by default
        target_index = 0

        # Use at most 300 observations
        n_shap = min(
            300,
            len(X_pool)
        )

        X_shap = X_pool[
            :n_shap
        ]

        explainer = shap.TreeExplainer(
            model
        )

        shap_values = explainer(
            X_shap
        )

        # Multi-output XGB handling
        values = shap_values.values

        if values.ndim == 3:

            values_t1 = values[
                :,
                :,
                target_index
            ]

        else:

            values_t1 = values

        mean_abs_shap = (
            np.mean(
                np.abs(
                    values_t1
                ),
                axis=0
            )
        )

        shap_df = pd.DataFrame(
            {
                "feature": FEATURES,
                "mean_abs_SHAP_t1":
                    mean_abs_shap
            }
        ).sort_values(
            "mean_abs_SHAP_t1",
            ascending=False
        )

        shap_df.to_csv(
            OUTPUT_DIR /
            f"SHAP_XGB_t1_{year}.csv",
            index=False
        )

except ImportError:

    print(
        "\nSHAP not installed. "
        "Skipping SHAP analysis."
    )


# ============================================================
# 45. FINAL METADATA
# ============================================================

metadata = {

    "study": {
        "topic":
            "Climate-induced domain shift and "
            "data-efficient model adaptation",
        "target_journal":
            "Applied Energy"
    },

    "datasets": {
        "current":
            str(CURRENT_PATH),
        "2050":
            str(FUTURE_2050_PATH),
        "2080":
            str(FUTURE_2080_PATH)
    },

    "schema": {
        "n_features":
            len(FEATURES),
        "features":
            FEATURES,
        "targets":
            TARGETS,
        "schema_identical":
            True
    },

    "future_climate": {
        "scenario":
            "SRES A2",
        "GCM":
            "HadCM3",
        "weather_tool":
            "CCWorldWeatherGen v1.9",
        "framework":
            "IPCC AR4 / SRES"
    },

    "experimental_design": {
        "current_test_size":
            CURRENT_TEST_SIZE,
        "future_test_size":
            FUTURE_TEST_SIZE,
        "future_fractions":
            FUTURE_FRACTIONS,
        "repeat_seeds":
            REPEAT_SEEDS,
        "validation_size":
            VALIDATION_SIZE
    },

    "models": [
        "Random Forest",
        "XGBoost",
        "DNN",
        "DNN Transfer Learning"
    ],

    "training_regimes": [
        "zero_shot",
        "future_only",
        "pooled_current_future",
        "transfer_learning",
        "full_future"
    ],

    "metrics": [
        "R2",
        "RMSE",
        "MAE",
        "Bias_MBE",
        "CV_RMSE_percent",
        "Spearman_Rho"
    ],

    "dnn": {
        "target_scaling":
            "StandardScaler",
        "feature_scaling":
            "StandardScaler",
        "early_stopping":
            True,
        "restore_best_weights":
            True,
        "source_learning_rate":
            DNN_SOURCE_LR,
        "transfer_learning_rate":
            DNN_ADAPT_LR,
        "scratch_learning_rate":
            DNN_SCRATCH_LR
    },

    "uncertainty": {
        "bootstrap_iterations":
            N_BOOTSTRAP,
        "confidence_interval":
            "95%"
    },

    "domain_shift": {
        "input_MMD":
            True,
        "target_distribution_shift":
            True,
        "PCA":
            True
    },

    "reproducibility": {
        "global_seed":
            GLOBAL_SEED,
        "tensorflow_determinism":
            True
    }
}


with open(
    OUTPUT_DIR /
    "pipeline_metadata.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )


# ============================================================
# 46. FINAL SUMMARY
# ============================================================

print(
    "\n\n======================================================"
)

print(
    "PIPELINE COMPLETED"
)

print(
    "======================================================"
)

print(
    f"\nResults directory:"
)

print(
    OUTPUT_DIR.resolve()
)

print(
    "\nMain files:"
)

for file in sorted(
    OUTPUT_DIR.glob("*")
):

    print(
        " -",
        file.name
    )


print(
    "\n======================================================"
)

print(
    "EXPERIMENTAL STRUCTURE"
)

print(
    "======================================================"
)

print(
    """
1. Current source model
2. Zero-shot future prediction
3. Future-only scratch training
4. Current + Future pooled training
5. DNN transfer learning
6. Full-future reference
7. Repeated future-data fractions
8. Fixed independent future test sets
9. R2 + RMSE + MAE + Bias + Spearman
10. Absolute simulation cost
11. Performance recovery
12. Bootstrap uncertainty
13. Input-space MMD
14. Target-distribution shift
15. PCA domain visualization
16. DNN freezing sensitivity
17. Transfer-learning vs scratch
18. Pooled vs future-only
"""
)

print(
    "\nDone."
)


Loading Current:
D:\UPM\Machine Learning\Scenario 1 Present Climate\Synthetic_data_sdv_dummies_1826_current climate.csv
Current shape: (1826, 112)

Loading 2050:
D:\UPM\Machine Learning\Scenario 2 2050\Synthetic_data_future_2050_processed.csv
2050 shape: (909, 112)

Loading 2080:
D:\UPM\Machine Learning\Scenario 2 2080\Synthetic_data_future_2080_processed.csv
2080 shape: (991, 112)
Schema verified: Current == 2050
Schema verified: Current == 2080

Number of input features: 107
Number of targets: 5

First features:
['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10_RoofwithInsulationBatt154mm']

Last features:
['P25_PTAC-DOAS', 'P25_PTHP', 'P25_PTHP-DOAS', 'P25_PackagedVAV', 'P25_UnitaryHeatPump', 'P25_VAVAir-CooledChillers', 'P25_VAVWater-CooledChillers', 'P25_VRF', 'P25_VRF-DOAS', 'P25_WaterToAirHeatPump']

Current duplicate check:
Duplicate complete rows: 0
Duplicate feature configurations: 0

2050 duplicate check:
Duplicate complete rows: 0
Duplicate feature configurations

ExplainerError: Additivity check failed in TreeExplainer! Please ensure the data matrix you passed to the explainer is the same shape that the model was trained on. If your data shape is correct then please report this on GitHub. Consider retrying with the feature_perturbation='interventional' option. This check failed because for one of the samples the sum of the SHAP values was 181.098984, while the model output was 181.094666. If this difference is acceptable you can set check_additivity=False to disable this check.

In [4]:
print("Results directory:")
print(OUTPUT_DIR.resolve())

print("\nAll result files:")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        print(f)

Results directory:
C:\Users\DELL PRECISION\Machine Learning\AppliedEnergy_ClimateDomainShift_Results

All result files:
AppliedEnergy_ClimateDomainShift_Results\aggregated_results_mean_std.csv
AppliedEnergy_ClimateDomainShift_Results\all_model_metrics.csv
AppliedEnergy_ClimateDomainShift_Results\bootstrap_confidence_intervals.csv
AppliedEnergy_ClimateDomainShift_Results\current_source_performance.csv
AppliedEnergy_ClimateDomainShift_Results\data_efficiency_R2_2050_t1.png
AppliedEnergy_ClimateDomainShift_Results\data_efficiency_R2_2050_t2.png
AppliedEnergy_ClimateDomainShift_Results\data_efficiency_R2_2050_t3.png
AppliedEnergy_ClimateDomainShift_Results\data_efficiency_R2_2050_t4.png
AppliedEnergy_ClimateDomainShift_Results\data_efficiency_R2_2050_t5.png
AppliedEnergy_ClimateDomainShift_Results\data_efficiency_R2_2080_t1.png
AppliedEnergy_ClimateDomainShift_Results\data_efficiency_R2_2080_t2.png
AppliedEnergy_ClimateDomainShift_Results\data_efficiency_R2_2080_t3.png
AppliedEnergy_Climat